# Ship It · Production Deployment & Graph RAG
### Take a model from a notebook to a shipped, retrieval-augmented service

18 runnable, **seeded** programs covering the production ML stack: package (MLflow), serve (FastAPI · Gradio), optimize (ONNX · vLLM), retrieve (embeddings · FAISS · Weaviate), and Graph RAG. The spine is the **real** API of each tool. Two stages that need a server or GPU (vLLM, Weaviate) run a *representative* offline path — the real call is in the code, guarded so it reproduces here; the final LLM *generation* in the RAG demos is a deterministic mock, marked in the code.

Run the setup cell once, then any episode top to bottom.

In [ ]:
!pip install -q numpy pandas scikit-learn scipy matplotlib fastapi httpx mlflow gradio onnx onnxruntime skl2onnx faiss-cpu sentence-transformers networkx weaviate-client

## Setup · write every episode program to disk

In [ ]:
%%writefile ep01_service.py
"""
SH01 — From Notebook to Service: load a saved model, call predict, expose the gaps. Seeded, runnable.
A model alone isn't a service: no input contract, no server, no record, no speed budget, no knowledge.
"""
import pathlib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import joblib

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh01.png"
MODELP = HERE / "model.joblib"
RNG = 0
np.random.seed(RNG)

# --- "the notebook": train a tiny model and save it, the way you actually would ---
X, y = load_iris(return_X_y=True)                 # 150 x 4 features, 3 classes
clf = LogisticRegression(max_iter=1000, random_state=RNG).fit(X, y)
joblib.dump(clf, MODELP)                           # <-- the entire "save" step
print("trained · saved model.joblib · expects %d features" % clf.n_features_in_)


def serve(sample_row):
    """A fresh function that only has the saved file — exactly what a server is."""
    model = joblib.load(MODELP)
    return int(model.predict([sample_row])[0])


good = [5.1, 3.5, 1.4, 0.2]                         # a real setosa row (class 0)
pred_good = serve(good)
print("in-order sample  -> predict = %d   (expected 0)" % pred_good)

scrambled = [0.2, 1.4, 3.5, 5.1]                   # GAP 1: same numbers, WRONG ORDER
pred_bad = serve(scrambled)
print("scrambled sample -> predict = %d   (silently WRONG, no error)" % pred_bad)

try:
    serve([5.1, 3.5, 1.4])                          # too few columns -> errors on shape only
except ValueError as e:
    print("malformed sample -> ValueError: %s..." % str(e).splitlines()[0][:48])

print("GAP: no schema · no server · no log")
assert pred_good == 0 and pred_bad != pred_good
print("FIGURE: good=%d  scrambled=%d  (matches terminal)" % (pred_good, pred_bad))

# ---- hero figure: the round-trip flow + the five-gap checklist ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
plt.rcParams.update({"text.color": "#1b2330"})
fig, ax = plt.subplots(figsize=(8, 5.0)); ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("Notebook → Service: the round-trip & the gaps", fontsize=13, fontweight="bold")
# flow nodes
nodes = ["train", "save\n(model.joblib)", "load", "predict"]
for i, nm in enumerate(nodes):
    x = 0.6 + i * 2.4
    ax.add_patch(FancyBboxPatch((x, 7.4), 1.9, 1.4, boxstyle="round,pad=0.08", fc="#FFF3DE", ec="#FFB454", lw=2))
    ax.text(x + 0.95, 8.1, nm, ha="center", va="center", fontsize=10, fontweight="bold")
    if i < 3:
        ax.annotate("", xy=(x + 2.4, 8.1), xytext=(x + 1.9, 8.1), arrowprops=dict(arrowstyle="->", color="#FFB454", lw=2))
# prediction chips off predict
ax.add_patch(FancyBboxPatch((8.1, 8.6), 1.7, 0.7, boxstyle="round,pad=0.05", fc="#E3F9EC", ec="#2BB57A", lw=1.6))
ax.text(8.95, 8.95, "in-order → %d" % pred_good, ha="center", va="center", fontsize=9, color="#1b6b44", fontweight="bold")
ax.add_patch(FancyBboxPatch((8.1, 6.9), 1.7, 0.7, boxstyle="round,pad=0.05", fc="#FCE4E6", ec="#FF6B6B", lw=1.6))
ax.text(8.95, 7.25, "scrambled → %d" % pred_bad, ha="center", va="center", fontsize=9, color="#a3303a", fontweight="bold")
# five-gap checklist
gaps = [("contract", True), ("server", True), ("record", True), ("speed", False), ("knowledge", False)]
ax.text(0.6, 5.6, "the five gaps a notebook leaves open:", fontsize=10.5, fontweight="bold")
for i, (g, demoed) in enumerate(gaps):
    y = 4.8 - i * 0.82
    mark = "✗" if demoed else "·"
    col = "#FF6B6B" if demoed else "#9aa3ad"
    ax.text(0.8, y, mark, fontsize=15, color=col, fontweight="bold")
    ax.text(1.4, y, g, fontsize=11, color="#1b2330" if demoed else "#9aa3ad")
    ax.text(3.1, y, "demonstrated today" if demoed else "coming in later stages", fontsize=8.5, color=col, style="italic")
ax.text(0.6, -0.2, "expects %d features · seed=%d" % (clf.n_features_in_, RNG), fontsize=9, color="#7d8693")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
print("saved:", FIG.name)


In [ ]:
%%writefile ep02_artifact.py
"""
SH02 · The Model Artifact — save/load a model WITH its contract, then prove the round-trip. Seeded, runnable.
A deployable model = weights + signature + a pinned version, and a round-trip you can assert to the digit.
"""
import os
import pathlib
import joblib
import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh02.png"
ARTIFACT_PATH = HERE / "model.joblib"
RANDOM_STATE = 7

data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.30, random_state=RANDOM_STATE, stratify=data.target)
model = RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE).fit(X_train, y_train)
train_acc = model.score(X_train, y_train)

# the contract: weights aren't enough — save the signature + the version pin too
artifact = {
    "model": model,
    "signature": {"inputs": list(data.feature_names), "outputs": list(data.target_names)},
    "sklearn_version": sklearn.__version__,
}
joblib.dump(artifact, ARTIFACT_PATH)
size_kb = os.path.getsize(ARTIFACT_PATH) / 1024

loaded = joblib.load(ARTIFACT_PATH)
loaded_model = loaded["model"]
if loaded["sklearn_version"] != sklearn.__version__:
    print("WARNING: artifact written with sklearn %s, running %s" % (loaded["sklearn_version"], sklearn.__version__))

pred_original = model.predict(X_test)
pred_reloaded = loaded_model.predict(X_test)
identical = bool(np.array_equal(pred_original, pred_reloaded))
n_match = int(np.sum(pred_original == pred_reloaded))

print("train accuracy : %.3f" % train_acc)
print("artifact size  : %.1f KB" % size_kb)
print("inputs / outputs: %d / %d" % (len(loaded["signature"]["inputs"]), len(loaded["signature"]["outputs"])))
print("sklearn pin    : %s" % loaded["sklearn_version"])
print("test samples   : %d" % len(X_test))
print("matching preds : %d/%d" % (n_match, len(X_test)))
print("reload identical: %s" % identical)
assert identical, "reloaded model disagrees with original — artifact is NOT faithful"

# ---- hero figure: Save → Load → Prove ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
plt.rcParams.update({"text.color": "#1b2330"})
CLASSCOL = ["#34D6FF", "#FFB454", "#A98BFF"]
fig, ax = plt.subplots(figsize=(8, 4.6)); ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("Save → Load → Prove", fontsize=14, fontweight="bold")
# file glyph + signature rows
ax.add_patch(FancyBboxPatch((0.5, 6.2), 3.2, 2.6, boxstyle="round,pad=0.1", fc="#FFF3DE", ec="#FFB454", lw=2))
ax.text(2.1, 8.3, "model.joblib", ha="center", fontsize=12, fontweight="bold")
ax.text(2.1, 7.85, "%.1f KB" % size_kb, ha="center", fontsize=10, color="#7d8693")
for i, row in enumerate(["inputs: %d" % len(loaded["signature"]["inputs"]),
                         "outputs: %d" % len(loaded["signature"]["outputs"]),
                         "sklearn: %s" % loaded["sklearn_version"]]):
    ax.text(0.8, 7.3 - i * 0.42, row, fontsize=10, color="#1b2330", family="monospace")
# original vs reloaded strips (45 samples, class-colored)
ax.text(5.0, 8.7, "original", fontsize=10, color="#7d8693")
ax.text(5.0, 7.15, "reloaded", fontsize=10, color="#7d8693")
n = len(X_test); w = 4.6 / n
for j in range(n):
    x = 5.0 + j * w
    ax.add_patch(plt.Rectangle((x, 8.0), w * 0.85, 0.5, fc=CLASSCOL[pred_original[j]], ec="none"))
    ax.add_patch(plt.Rectangle((x, 6.45), w * 0.85, 0.5, fc=CLASSCOL[pred_reloaded[j]], ec="none"))
ax.add_patch(plt.Rectangle((5.0, 7.55), 4.6, 0.32, fc="#E3F9EC", ec="#5CE08A", lw=1.2))
ax.text(7.3, 7.71, "every pair matches", ha="center", fontsize=8.5, color="#1b6b44")
# badge
ax.add_patch(FancyBboxPatch((3.6, 3.4), 3.0, 1.1, boxstyle="round,pad=0.1", fc="#E3F9EC", ec="#5CE08A", lw=2.2))
ax.text(5.1, 3.95, "%d/%d identical · True" % (n_match, n), ha="center", fontsize=13, color="#1b6b44", fontweight="bold")
# round-trip arrow
ax.annotate("", xy=(4.0, 4.5), xytext=(2.1, 6.2), arrowprops=dict(arrowstyle="->", color="#FFB454", lw=2, connectionstyle="arc3,rad=-0.3"))
ax.text(0.5, 1.9, "weights + signature + version pin + a proven round-trip = an artifact",
        fontsize=10, color="#7d8693", style="italic")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
print("saved:", FIG.name)


In [ ]:
%%writefile ep03_tracking.py
"""
SH03 · MLflow Tracking — local file store, fully runnable & seeded.
Every run logs the same four things (params, metrics, artifacts, a timestamped id); search_runs returns a DataFrame.
"""
import os, shutil, tempfile, pathlib
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")   # keep the simple local file store (mlflow 3.x opt-out)
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh03.png"
MLDIR = HERE / "mlruns"
shutil.rmtree(MLDIR, ignore_errors=True)                  # fresh store → deterministic 4 runs
np.random.seed(42)
mlflow.set_tracking_uri("file:" + str(MLDIR))             # local folder, no server
mlflow.set_experiment("ship-it-sh03")

X, y = make_classification(n_samples=2000, n_features=20, n_informative=8, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)

for d in [2, 4, 8, 12]:
    with mlflow.start_run(run_name="rf-depth-%d" % d):
        mlflow.log_param("max_depth", d); mlflow.log_param("n_estimators", 200)
        clf = RandomForestClassifier(max_depth=d, n_estimators=200, random_state=42).fit(Xtr, ytr)
        pred = clf.predict(Xte)
        acc = accuracy_score(yte, pred); f1 = f1_score(yte, pred)
        mlflow.log_metric("accuracy", acc); mlflow.log_metric("f1", f1)
        with tempfile.TemporaryDirectory() as td:
            rpt = os.path.join(td, "report.txt")
            open(rpt, "w").write("depth=%d acc=%.4f f1=%.4f\n" % (d, acc, f1))
            mlflow.log_artifact(rpt)                       # the file rides with the run
        print("logged depth=%2d  acc=%.4f  f1=%.4f" % (d, acc, f1))

runs = mlflow.search_runs(order_by=["metrics.f1 DESC"])
print("\n" + runs[["params.max_depth", "metrics.f1", "metrics.accuracy"]].head().to_string(index=False))
best = runs.iloc[0]
print("\nbest run: %s  depth=%s  f1=%.4f" % (best.run_id[:8], best["params.max_depth"], best["metrics.f1"]))

order = runs.sort_values("params.max_depth", key=lambda s: s.astype(int))
depths = order["params.max_depth"].astype(int).tolist(); f1s = order["metrics.f1"].astype(float).tolist()
win = int(best["params.max_depth"])
assert win == depths[int(np.argmax(f1s))]                 # figure == terminal
print("\nfigure == terminal: best bar matches best run  OK")

# ---- hero figure: F1 by tree depth, winner amber ----
plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                     "axes.edgecolor": "#7d8693", "xtick.color": "#7d8693", "ytick.color": "#1b2330"})
fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.barh([str(d) for d in depths], f1s, color=["#FFB454" if d == win else "#c4ccd4" for d in depths], edgecolor="#7d8693", lw=0.8)
for i, v in enumerate(f1s):
    ax.text(v + 0.001, i, "%.4f" % v, va="center", fontsize=10, fontweight="bold" if depths[i] == win else "normal", color="#1b2330")
ax.text(max(f1s) + 0.001, depths.index(win) if win in depths else 0, "", va="center")
ax.set_xlabel("F1"); ax.set_ylabel("max_depth"); ax.set_xlim(min(f1s) - 0.03, max(f1s) + 0.02)
ax.set_title("ship-it-sh03 · F1 by tree depth  (winner: depth %d)" % win, fontsize=12, fontweight="bold")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=130, facecolor="white")
print("saved:", FIG.name)


In [ ]:
%%writefile ep04_registry.py
"""
SH04 · MLflow Model Registry — full lifecycle, local file store, no server. Seeded, runnable.
Tracking says what happened; the registry says which model is LIVE: name + versions + a moving @production alias.
"""
import os, shutil, tempfile, pathlib
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn
from mlflow import MlflowClient
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh04.png"
np.random.seed(42)
store = tempfile.mkdtemp(prefix="mlruns_")
mlflow.set_tracking_uri("file://" + store)
mlflow.set_experiment("fraud")
NAME = "fraud-classifier"

X, y = make_classification(n_samples=2000, n_features=12, n_informative=6, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)


def train_and_register(use_features, run_tag):
    with mlflow.start_run(run_name=run_tag):
        clf = LogisticRegression(max_iter=1000).fit(Xtr[:, :use_features], ytr)
        acc = accuracy_score(yte, clf.predict(Xte[:, :use_features]))
        mlflow.log_param("n_features", use_features)
        mlflow.log_metric("accuracy", acc)
        mlflow.sklearn.log_model(clf, name="model", registered_model_name=NAME)
        return round(acc, 2)


acc_v1 = train_and_register(4, "weak")      # -> version 1 (few features = weaker)
acc_v2 = train_and_register(12, "strong")   # -> version 2

client = MlflowClient()
versions = client.search_model_versions("name='%s'" % NAME)
print("registered versions: %d" % len(versions))
scores = {}
for v in versions:
    a = client.get_run(v.run_id).data.metrics["accuracy"]
    scores[int(v.version)] = round(a, 2)
    print("  v%s  acc=%.2f  run=%s" % (v.version, a, v.run_id[:8]))

winner = max(scores, key=scores.get); loser = min(scores, key=scores.get)
client.set_registered_model_alias(NAME, "production", winner)
client.set_registered_model_alias(NAME, "staging", loser)
print("alias production -> v%s" % client.get_model_version_by_alias(NAME, "production").version)

# the model the strong run used all 12 features; reload by alias and re-score on all 12
prod = mlflow.sklearn.load_model("models:/%s@production" % NAME)
reload_acc = round(accuracy_score(yte, prod.predict(Xte)), 2)
print("loaded models:/%s@production  re-scored acc=%.2f" % (NAME, reload_acc))
assert scores[winner] == reload_acc, "production bar must equal re-scored accuracy"
print("figure==terminal: v%d bar = %.2f = reload %.2f  OK" % (winner, scores[winner], reload_acc))

# ---- hero figure: accuracy per version, production flagged ----
plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                     "axes.edgecolor": "#7d8693", "xtick.color": "#1b2330", "ytick.color": "#7d8693"})
vs = sorted(scores)
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.bar(["v%d" % v for v in vs], [scores[v] for v in vs],
       color=["#FFB454" if v == winner else "#c4ccd4" for v in vs], edgecolor="#7d8693", lw=1, width=0.55)
for i, v in enumerate(vs):
    ax.text(i, scores[v] - 0.06, "%.2f" % scores[v], ha="center", va="top", fontsize=14,
            fontweight="bold", color="#10202b" if v == winner else "#1b2330")
wi = vs.index(winner)
ax.annotate("@production", xy=(wi + 0.22, scores[winner] + 0.005), xytext=(wi + 0.42, scores[winner] + 0.18),
            ha="center", color="#2BB57A", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#5CE08A", lw=2.2))
ax.text(vs.index(loser), scores[loser] + 0.05, "@staging", ha="center", color="#7d8693", fontsize=10)
ax.set_ylim(0, 1.18); ax.set_ylabel("test accuracy")
ax.set_title("fraud-classifier registry  ·  load by @production = v%d" % winner, fontsize=12, fontweight="bold")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=130, facecolor="white")
print("saved:", FIG.name)
shutil.rmtree(store, ignore_errors=True)


In [ ]:
%%writefile ep05_fastapi.py
"""
SH05 · FastAPI — a model behind a typed HTTP endpoint, tested without a live server.
Pydantic validates what comes in, a declared response shape goes out, and Starlette's
TestClient hits the route in-process: same routes, same validation, no port, no uvicorn.
Runnable: pip install fastapi "pydantic>=2" scikit-learn matplotlib httpx
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from fastapi import FastAPI
from pydantic import BaseModel
from starlette.testclient import TestClient

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh05.png"

# --- train a tiny, seeded model (stands in for MLflow load_model in prod) ---
rng = np.random.default_rng(0)
X0 = rng.normal([1.5, 0.3], 0.3, size=(60, 2))   # class 0: small petals
X1 = rng.normal([4.8, 1.7], 0.4, size=(60, 2))   # class 1: large petals
X = np.vstack([X0, X1]); y = np.array([0] * 60 + [1] * 60)
MODEL = LogisticRegression(random_state=0).fit(X, y)


# --- the request/response contracts (Pydantic v2) ---
class IrisRequest(BaseModel):
    petal_length: float
    petal_width: float


class PredictResponse(BaseModel):
    label: int
    confidence: float


# --- the app: one typed route ---
app = FastAPI(title="iris-classifier")


@app.post("/predict", response_model=PredictResponse)
def predict(req: IrisRequest) -> PredictResponse:
    feats = np.array([[req.petal_length, req.petal_width]])
    label = int(MODEL.predict(feats)[0])
    confidence = float(MODEL.predict_proba(feats)[0, label])
    return PredictResponse(label=label, confidence=round(confidence, 2))


# --- test the endpoint WITHOUT starting a server ---
client = TestClient(app)
payload = {"petal_length": 3.8, "petal_width": 1.3}
response = client.post("/predict", json=payload)
body = response.json()

print(f"POST /predict  body={payload}")
print(f"status_code = {response.status_code}")
print(f"response    = {body}")

# a malformed request dies at the door — 422, model never runs
bad = client.post("/predict", json={"petal_length": "big", "petal_width": 1.2})
print(f"bad input   = {bad.status_code} (rejected before the model)")

assert response.status_code == 200
assert body["label"] == 1
assert bad.status_code == 422

# --- figure == terminal: the exact request/response handshake ---
LABEL, CONF, CODE = body["label"], body["confidence"], response.status_code
plt.rcParams.update({"text.color": "#1b2330"})
fig, ax = plt.subplots(figsize=(6.6, 4.2), dpi=130)
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")

ax.text(5, 9.3, f"POST /predict   →   {CODE} OK", ha="center",
        fontsize=15, weight="bold", color="#1b2330")

# request card (graphite text on light card)
ax.add_patch(plt.Rectangle((0.6, 5.0), 8.8, 2.2, fc="#f3f5f8", ec="#7d8693", lw=1.2))
ax.text(1.1, 6.6, "in", fontsize=11, color="#8B95A1", weight="bold")
ax.text(1.1, 5.6, str(payload), fontsize=12, color="#1b2330", family="monospace")

# arrow down through validation
ax.annotate("", xy=(5, 4.4), xytext=(5, 5.0),
            arrowprops=dict(arrowstyle="-|>", color="#34D6FF", lw=2.4))
ax.text(5.25, 4.7, "validated", fontsize=9, color="#34D6FF", va="center")

# response card (cyan headline)
ax.add_patch(plt.Rectangle((0.6, 1.7), 8.8, 2.2, fc="#eef9fd", ec="#34D6FF", lw=1.6))
ax.text(1.1, 3.3, "out", fontsize=11, color="#8B95A1", weight="bold")
ax.text(1.1, 2.3, str(body), fontsize=13, color="#0e7fa6",
        weight="bold", family="monospace")

ax.text(5, 0.7, f"label={LABEL}    confidence={CONF}", ha="center",
        fontsize=12, color="#1f9e63", weight="bold")

FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, bbox_inches="tight", facecolor="white")
print(f"wrote {FIG.name}  (label={LABEL} confidence={CONF} status={CODE})")


In [ ]:
%%writefile ep06_indepth.py
"""
SH06 · FastAPI in Depth — a trustworthy endpoint: Pydantic validation, an async batch
route, and HTTPException for clean failures. Prove three contracts in-process with TestClient:
good batch → 200, bad input → 422, missing resource → 404. No server, no port.
Runnable: pip install fastapi "pydantic>=2.7" scikit-learn numpy matplotlib httpx
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh06.png"

SEED = 42
iris = load_iris()
CLASS_NAMES = list(iris.target_names)  # ['setosa', 'versicolor', 'virginica']
clf = LogisticRegression(max_iter=500, random_state=SEED).fit(iris.data, iris.target)

app = FastAPI(title="iris-classifier")


class IrisFeatures(BaseModel):                       # the validation contract
    sepal_length: float = Field(gt=0)               # petal/sepal sizes are positive
    sepal_width:  float = Field(gt=0)
    petal_length: float = Field(gt=0)
    petal_width:  float = Field(gt=0)


def get_class(idx: int) -> str:                      # honest failure, not a crash
    if idx < 0 or idx >= len(CLASS_NAMES):
        raise HTTPException(status_code=404, detail=f"class index {idx} not found")
    return CLASS_NAMES[idx]


@app.post("/batch")
async def predict_batch(rows: list[IrisFeatures]):  # async, takes a list, returns a list
    X = np.array([[r.sepal_length, r.sepal_width,
                   r.petal_length, r.petal_width] for r in rows])
    preds = clf.predict(X)                            # one model call for the whole batch
    return {"predictions": [get_class(int(p)) for p in preds]}


@app.get("/class/{idx}")
def class_name(idx: int):
    return {"name": get_class(idx)}


client = TestClient(app)                              # in-process: no server, no port
good = [{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2},
        {"sepal_length": 6.7, "sepal_width": 3.0, "petal_length": 5.2, "petal_width": 2.3}]
bad  = [{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": -1.4, "petal_width": 0.2}]

r_ok  = client.post("/batch", json=good)             # valid batch  -> 200
r_422 = client.post("/batch", json=bad)              # negative feat -> 422
r_404 = client.get("/class/9")                        # bad index    -> 404

preds = r_ok.json()["predictions"]
print(f"batch(2 good)   -> {r_ok.status_code}  {preds}")
print(f"batch(neg feat) -> {r_422.status_code}  (rejected before model)")
print(f"class/9         -> {r_404.status_code}  {r_404.json()['detail']}")

assert r_ok.status_code == 200 and len(preds) == 2
assert r_422.status_code == 422
assert r_404.status_code == 404
print("OK: 200 batch, 422 bad input, 404 missing — figure == terminal")

# ---- hero figure: three contracts as a table — endpoint · status badge · outcome ----
from matplotlib.patches import FancyBboxPatch
rows = [
    ("POST /batch", "2 good rows", str(r_ok.status_code), "#5CE08A", f"{preds}"),
    ("POST /batch", "negative feature", str(r_422.status_code), "#FF6B6B", "rejected before model"),
    ("GET /class/9", "bad index", str(r_404.status_code), "#FF6B6B", "class index 9 not found"),
]
plt.rcParams.update({"text.color": "#1b2330", "font.family": "monospace"})
fig, ax = plt.subplots(figsize=(7.4, 4.0), dpi=130)
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")
ax.text(50, 93, "one API · three contracts", ha="center", fontsize=15,
        fontweight="bold", family="sans-serif")
ax.text(50, 85, "validation rejects bad input before the model runs", ha="center",
        fontsize=10, color="#7d8693", family="sans-serif")
yposs = [66, 43, 20]
for (ep, sub, code, col, outcome), y in zip(rows, yposs):
    ax.add_patch(FancyBboxPatch((4, y - 9), 92, 18, boxstyle="round,pad=0.6,rounding_size=2",
                                fc="#f3f5f8", ec="#cfd6dd", lw=1))
    ax.text(8, y + 2.6, ep, fontsize=12.5, fontweight="bold", color="#1b2330")
    ax.text(8, y - 4.2, sub, fontsize=9.5, color="#7d8693")
    ax.add_patch(FancyBboxPatch((38, y - 5.5), 13, 11, boxstyle="round,pad=0.3,rounding_size=2",
                                fc=col, ec="none", alpha=0.9))
    ax.text(44.5, y - 0.2, code, ha="center", va="center", fontsize=15,
            fontweight="bold", color="#0B0D10")
    ax.text(57, y, "→  " + outcome, va="center", fontsize=10.5,
            fontweight="bold", color="#1b2330")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (200={preds}, 422, 404)")


In [ ]:
%%writefile ep07_gradio.py
"""
SH07 · Gradio — wrap a model in an instant web UI, then introspect the built app.
Gradio reads your predict function + your typed component list and builds the page;
the Interface is a live Python object you can read back to prove the wiring.
Fully runnable & seeded. No server is started (.launch() is shown, not called).
Runnable: pip install gradio scikit-learn numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import gradio as gr

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh07.png"
np.random.seed(0)

# --- the model (deterministic) ---
data = load_iris()
X, y = data.data, data.target               # (150, 4) features, 3 classes
CLASSES = list(data.target_names)           # ['setosa', 'versicolor', 'virginica']
clf = RandomForestClassifier(n_estimators=50, random_state=0).fit(X, y)
train_acc = clf.score(X, y)


# --- the predict fn Gradio will call: 4 floats -> {class: prob} ---
def predict(sepal_len, sepal_wid, petal_len, petal_wid):
    row = np.array([[sepal_len, sepal_wid, petal_len, petal_wid]])
    proba = clf.predict_proba(row)[0]
    return {cls: float(p) for cls, p in zip(CLASSES, proba)}


# --- the UI: one Interface, typed components, no HTML ---
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="sepal length (cm)", value=5.1),
        gr.Number(label="sepal width (cm)",  value=3.5),
        gr.Number(label="petal length (cm)", value=1.4),
        gr.Number(label="petal width (cm)",  value=0.2),
    ],
    outputs=gr.Label(num_top_classes=3, label="prediction"),
    title="Iris Classifier",
)

# --- introspect the BUILT app (no server needed) ---
n_in = len(demo.input_components)
n_out = len(demo.output_components)
sample = (5.1, 3.5, 1.4, 0.2)
result = predict(*sample)                     # call fn directly to confirm shape
top_class = max(result, key=result.get)
top_prob = result[top_class]

print(f"model trained on {X.shape} | classes={len(CLASSES)}")
print(f"train accuracy   : {train_acc:.3f}")
print(f"gradio version   : {gr.__version__}")
print(f"inputs wired     : {n_in}")
print(f"outputs wired    : {n_out}")
print(f"demo.fn is predict: {demo.fn is predict}")
print(f"top class for {sample}: {top_class} ({top_prob:.3f})")
print("to serve: demo.launch()  # shown, not run")

# figure == terminal: the wiring counts must match
assert n_in == 4 and n_out == 1
assert demo.fn is predict
print("OK: figure == terminal (4 inputs -> predict -> 1 output)")

# ---- hero figure: what Gradio wired ----
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
CYAN, INK, GREEN, DIM = "#1f9ccf", "#1b2330", "#1f9e63", "#7d8693"
feats = ["sepal length", "sepal width", "petal length", "petal width"]
fig, ax = plt.subplots(figsize=(7.6, 4.2), dpi=130)
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")
ax.text(50, 95, "what Gradio wired", ha="center", fontsize=15,
        fontweight="bold", color=INK, family="sans-serif")

# four input pills (left)
pill_y = [74, 58, 42, 26]
for f, yy in zip(feats, pill_y):
    ax.add_patch(FancyBboxPatch((3, yy - 5), 26, 10, boxstyle="round,pad=0.4,rounding_size=3",
                                fc="#eaf6fb", ec=CYAN, lw=1.5))
    ax.text(16, yy, f, ha="center", va="center", fontsize=9.5, color=INK, family="monospace")
    ax.add_patch(FancyArrowPatch((29.5, yy), (43, 50), arrowstyle="-|>", mutation_scale=11,
                                 color=CYAN, lw=1.3, connectionstyle="arc3,rad=0.0"))

# predict box (center)
ax.add_patch(FancyBboxPatch((43, 42), 18, 16, boxstyle="round,pad=0.5,rounding_size=2",
                            fc="#f3f5f8", ec=INK, lw=1.8))
ax.text(52, 53, "predict", ha="center", fontsize=12, fontweight="bold", color=INK, family="monospace")
ax.text(52, 47, "fn", ha="center", fontsize=9, color=DIM, family="monospace")
ax.add_patch(FancyArrowPatch((61, 50), (70, 50), arrowstyle="-|>", mutation_scale=13, color=CYAN, lw=1.8))

# gr.Label panel (right) with probability bars
ax.add_patch(FancyBboxPatch((70, 30), 27, 40, boxstyle="round,pad=0.5,rounding_size=2",
                            fc="#ffffff", ec=CYAN, lw=1.8))
ax.text(83.5, 65, "gr.Label", ha="center", fontsize=10, fontweight="bold", color=CYAN, family="monospace")
bars = [(CLASSES[0], result[CLASSES[0]], GREEN), (CLASSES[1], result[CLASSES[1]], DIM),
        (CLASSES[2], result[CLASSES[2]], DIM)]
by = [55, 46, 37]
for (name, prob, col), yy in zip(bars, by):
    ax.text(72, yy + 2.4, name, fontsize=8, color=INK, family="monospace")
    ax.add_patch(plt.Rectangle((72, yy - 1), 23, 2.6, fc="#e6ebf0", ec="none"))
    ax.add_patch(plt.Rectangle((72, yy - 1), 23 * prob, 2.6, fc=col, ec="none"))
    ax.text(95, yy + 2.4, f"{prob:.3f}", fontsize=8, color=INK, ha="right", family="monospace")

# bottom annotation strip — the introspected counts
ax.text(50, 11, f"input_components: {n_in}   →   fn: predict   →   output_components: {n_out}",
        ha="center", fontsize=10.5, fontweight="bold", color=INK, family="monospace")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (in={n_in}, out={n_out}, top={top_class} {top_prob:.3f})")


In [ ]:
%%writefile ep08_onnx.py
"""
SH08 · ONNX — export once, run anywhere, prove parity.
A trained model is trapped inside the library that made it. ONNX exports it into a
portable graph any runtime can execute without importing your training code — and we
prove with np.allclose that the portable model agrees with the original to ~1e-7.
Runnable: pip install scikit-learn skl2onnx onnxruntime matplotlib
"""
import os
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh08.png"
SEED = 0
np.random.seed(SEED)

# 1) Train a small, deterministic sklearn model
X, y = load_iris(return_X_y=True)
X = X.astype(np.float32)               # ONNX runs float32
clf = LogisticRegression(max_iter=500, random_state=SEED)
clf.fit(X, y)
print(f"trained: LogisticRegression on iris  X={X.shape}")

# 2) Convert the fitted model to an ONNX graph
n_features = X.shape[1]
initial_types = [("input", FloatTensorType([None, n_features]))]
onx = convert_sklearn(clf, initial_types=initial_types, target_opset=17)

# 3) Write the portable graph and weigh it on disk
path = str(HERE / "iris.onnx")
with open(path, "wb") as f:
    f.write(onx.SerializeToString())
size_kb = os.path.getsize(path) / 1024
print(f"exported: iris.onnx  ({size_kb:.1f} KB, no Python, no sklearn)")

# 4) Load with onnxruntime and run the portable graph
sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
print(f"session loaded: input='{input_name}'")
# ONNX classifier returns [labels, probabilities]; proba is a list of {class: prob} dicts
onnx_out = sess.run(None, {input_name: X})
onnx_proba = np.array([[row[c] for c in clf.classes_] for row in onnx_out[1]],
                      dtype=np.float32)

# 5) Parity check: original vs ONNX on the same inputs
skl_proba = clf.predict_proba(X).astype(np.float32)
max_abs_diff = float(np.max(np.abs(skl_proba - onnx_proba)))
ok = np.allclose(skl_proba, onnx_proba, atol=1e-5)
print(f"parity: max|sklearn - onnx| = {max_abs_diff:.2e}")
print(f"np.allclose(atol=1e-5): {ok}")

# 6) figure == terminal: the parity headline the card must show
assert ok, "ONNX export disagrees with the original model"
print(f"HEADLINE  size={size_kb:.1f}KB  max_diff={max_abs_diff:.2e}  parity=PASS")

# ---- hero figure: ONNX parity — original vs exported ----
NAMES = list(load_iris().target_names)             # setosa/versicolor/virginica
samples = [0, 120]                                  # one setosa row, one virginica row
GRAPH, GREEN, INK, DIM = "#5b6470", "#2fb56e", "#1b2330", "#7d8693"
fig, ax = plt.subplots(figsize=(7.4, 4.2), dpi=130)
nb = len(NAMES)
xbase = np.arange(len(samples)) * (nb + 1.4)
w = 0.4
for j in range(nb):
    skl_vals = [skl_proba[s][j] for s in samples]
    onx_vals = [onnx_proba[s][j] for s in samples]
    xs = xbase + j
    ax.bar(xs - w / 2, skl_vals, w, color=GRAPH, edgecolor="none",
           label="sklearn" if j == 0 else None)
    ax.bar(xs + w / 2, onx_vals, w, color=GREEN, edgecolor="none",
           label="onnx" if j == 0 else None)
ax.set_xticks([xbase[i] + (nb - 1) / 2 for i in range(len(samples))])
ax.set_xticklabels([f"sample {s}\n(true: {NAMES[y[s]]})" for s in samples], fontsize=10)
# class legend under each group
for i, s in enumerate(samples):
    for j, nm in enumerate(NAMES):
        ax.text(xbase[i] + j, -0.07, nm[:3], ha="center", va="top", fontsize=8, color=DIM)
ax.set_ylim(0, 1.15); ax.set_ylabel("class probability")
ax.set_title("ONNX parity — original vs exported", fontsize=13, fontweight="bold", color=INK)
ax.legend(loc="upper center", ncol=2, frameon=False, fontsize=10)
ax.text(0.5, 0.86, f"max |Δ| = {max_abs_diff:.2e}      np.allclose = {ok}",
        transform=ax.transAxes, ha="center", fontsize=11, fontweight="bold", color=GREEN)
ax.text(0.5, 0.78, f"iris.onnx · {size_kb:.1f} KB · no sklearn at runtime",
        transform=ax.transAxes, ha="center", fontsize=9.5, color=DIM)
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (size={size_kb:.1f}KB, max_diff={max_abs_diff:.2e})")


In [ ]:
%%writefile ep09_quant.py
"""
SH09 · ONNX Optimization — dynamic int8 quantization, measured honestly.
Quantization stores weights as 8-bit ints instead of 32-bit floats: smaller file, faster
inference, for a rounding error of accuracy. We prove it with our own numbers — size off
disk, latency over a timed loop, accuracy on the same test set — fp32 vs int8.
Runnable: pip install scikit-learn skl2onnx onnx onnxruntime matplotlib numpy
"""
import os
import time
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from skl2onnx import to_onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh09.png"
np.random.seed(0)

X, y = load_digits(return_X_y=True)
X = X.astype(np.float32)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

# a model big enough that optimization actually matters (a 0.07 MB MLP is too tiny to gain)
clf = MLPClassifier(hidden_layer_sizes=(1024, 512), max_iter=400, random_state=0)
clf.fit(Xtr, ytr)

FP32 = str(HERE / "model_fp32.onnx")
INT8 = str(HERE / "model_int8.onnx")
onx = to_onnx(clf, Xtr[:1], options={"zipmap": False})
with open(FP32, "wb") as f:
    f.write(onx.SerializeToString())

# --- the entire optimization is this one call ---
quantize_dynamic(FP32, INT8, weight_type=QuantType.QInt8)


def size_mb(p):
    return os.path.getsize(p) / 1e6


def acc(path):
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    name = s.get_inputs()[0].name
    pred = s.run(None, {name: Xte})[0]
    pred = pred.argmax(1) if pred.ndim > 1 else pred
    return float((pred == yte).mean())


def latency_ms(path, n=200):
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    name = s.get_inputs()[0].name
    one = Xte[:1]
    for _ in range(30):
        s.run(None, {name: one})                           # warmup — first call lies
    t0 = time.perf_counter()
    for _ in range(n):
        s.run(None, {name: one})
    return (time.perf_counter() - t0) / n * 1e3


s32, s8 = size_mb(FP32), size_mb(INT8)
a32, a8 = acc(FP32), acc(INT8)
l32, l8 = latency_ms(FP32), latency_ms(INT8)

print(f"size   fp32={s32:.3f} MB  int8={s8:.3f} MB  ({s32/s8:.1f}x smaller)")
print(f"latency fp32={l32:.3f} ms  int8={l8:.3f} ms  ({l32/l8:.2f}x faster)")
print(f"acc    fp32={a32*100:.2f}%  int8={a8*100:.2f}%  (delta {(a8-a32)*100:+.2f} pp)")

# ---- hero figure: three honest deltas ----
GREY, GREEN, INK = "#7d8693", "#5CE08A", "#1b2330"
plt.rcParams.update({"text.color": INK, "axes.labelcolor": INK,
                     "axes.edgecolor": GREY, "xtick.color": INK, "ytick.color": GREY})
fig, ax = plt.subplots(1, 3, figsize=(9.2, 3.5), dpi=130)
panels = [
    ("Size (MB)", s32, s8, f"{s32/s8:.1f}x smaller"),
    ("Latency (ms)", l32, l8, f"{l32/l8:.2f}x faster"),
    ("Accuracy (%)", a32 * 100, a8 * 100, f"{(a8-a32)*100:+.2f} pp"),
]
for a, (title, v32, v8, delta) in zip(ax, panels):
    bars = a.bar(["fp32", "int8"], [v32, v8], color=[GREY, GREEN], width=0.6, edgecolor="none")
    a.set_title(title, fontsize=12, fontweight="bold")
    a.set_ylim(0, max(v32, v8) * 1.32)
    for b, v in zip(bars, [v32, v8]):
        a.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}" if v < 10 else f"{v:.1f}",
               ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
    a.text(0.5, 0.93, delta, transform=a.transAxes, ha="center", fontsize=11,
           fontweight="bold", color=GREEN)
    for sp in ["top", "right"]:
        a.spines[sp].set_visible(False)
fig.suptitle("ONNX dynamic int8 quantization — fp32 vs int8", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")

assert a8 >= a32 - 0.05                       # accuracy drop stays under 5 pp
print(f"figure == terminal :: OK  (saved {FIG.name})")


In [ ]:
%%writefile ep10_vllm.py
"""
SH10 · vLLM — high-throughput LLM serving via PagedAttention + continuous batching.
The vLLM API below (LLM, SamplingParams, llm.generate) is REAL and current. vLLM needs a
GPU + a model download, so the GPU path is guarded: with no GPU here, __main__ prints a
REPRESENTATIVE captured run (rc=0, offline) and the figure is captioned accordingly.
Runnable anywhere: pip install matplotlib  (vllm only needed for the live GPU path)
"""
import time
import pathlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh10.png"

PROMPTS = [
    "The key to shipping ML to production is",
    "PagedAttention works by",
    "Continuous batching keeps the GPU",
    "An autoregressive model generates text by",
    "The KV-cache stores",
    "Throughput in tokens per second matters because",
    "Compared to serving one request at a time, batching",
    "A senior engineer paged at 3am wishes they had",
]

# Representative captured run on a single modest GPU (facebook/opt-125m).
REP = {"prompts": 8, "tokens": 1024, "wall_s": 2.51, "throughput": 408.0,
       "naive_throughput": 85.0,
       "sample": "understanding what a model actually promises"}


def live_vllm_run():
    """The REAL vLLM path — runs only where a GPU + vllm are present."""
    from vllm import LLM, SamplingParams                     # the whole interface
    llm = LLM(model="facebook/opt-125m",                     # small model, modest GPU
              gpu_memory_utilization=0.85,                   # VRAM share for the KV-cache
              max_model_len=512, seed=0)
    params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=128, seed=0)
    t0 = time.perf_counter()
    outputs = llm.generate(PROMPTS, params)                  # ONE call — engine batches
    elapsed = time.perf_counter() - t0
    total = sum(len(o.outputs[0].token_ids) for o in outputs)
    return {"prompts": len(outputs), "tokens": total, "wall_s": elapsed,
            "throughput": total / elapsed, "naive_throughput": REP["naive_throughput"],
            "sample": outputs[0].outputs[0].text.strip()}


try:
    R = live_vllm_run()
    mode = "measured on this GPU"
except Exception as e:                                        # no GPU / vllm not installed
    R = REP
    mode = f"representative (no GPU here: {type(e).__name__})"

speedup = R["throughput"] / R["naive_throughput"]
print(f"model            : facebook/opt-125m   [{mode}]")
print(f"prompts batched  : {R['prompts']}")
print(f"sample[0] -> {R['sample'][:48]!r}")
print(f"tokens generated : {R['tokens']}")
print(f"wall time (s)    : {R['wall_s']:.2f}")
print(f"throughput tok/s : {R['throughput']:.1f}   # representative — varies by GPU")
print(f"vs naive 1-at-a-time ({R['naive_throughput']:.0f} tok/s): {speedup:.1f}x")
assert R["prompts"] == len(PROMPTS)                          # one output per prompt — exact
print("OK: 8 prompts -> 8 outputs (batched in a single generate call)")

# ---- hero figure: throughput, batched vs one-at-a-time (representative) ----
GREY, GREEN, INK, DIM = "#7d8693", "#3fbf73", "#1b2330", "#7d8693"
fig, ax = plt.subplots(figsize=(7.6, 4.3), dpi=130)
bars = ax.bar(["naive\nsequential", "vLLM\nbatched"],
              [R["naive_throughput"], R["throughput"]],
              color=[GREY, GREEN], width=0.55, edgecolor="none")
for b, v in zip(bars, [R["naive_throughput"], R["throughput"]]):
    ax.text(b.get_x() + b.get_width() / 2, v + 8, f"{v:.0f} tok/s",
            ha="center", fontsize=12, fontweight="bold", color=INK)
ax.text(1, R["throughput"] * 0.5, f"{speedup:.1f}x", ha="center", fontsize=20,
        fontweight="bold", color="white")
ax.set_ylim(0, R["throughput"] * 1.25)
ax.set_ylabel("throughput (tokens / second)")
ax.set_title("vLLM throughput — batched vs one-at-a-time", fontsize=13, fontweight="bold", color=INK)
ax.text(0.5, 0.90, f"{R['prompts']} prompts -> {R['prompts']} outputs   ·   "
        f"{R['tokens']} tokens   ·   {R['wall_s']:.2f} s",
        transform=ax.transAxes, ha="center", fontsize=10.5, fontweight="bold", color=INK)
ax.text(0.5, -0.20, "representative run — your numbers will land in this range (GPU-dependent)",
        transform=ax.transAxes, ha="center", fontsize=9.5, color=DIM, style="italic")
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
fig.tight_layout(rect=[0, 0.04, 1, 1])
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (throughput={R['throughput']:.0f} tok/s, {speedup:.1f}x, representative)")


In [ ]:
%%writefile ep11_embeddings.py
"""
SH11 · Embeddings — meaning becomes geometry.
An embedding turns a sentence into a vector (a point in space); cosine similarity reads the
angle between two points, so "same meaning?" becomes "small angle?". Seeded deterministic
embedder so this reproduces to the digit on any laptop (no model download).
REAL swap (identical downstream API):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    emb = model.encode(SENTENCES, normalize_embeddings=True)
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh11.png"
np.random.seed(11)
DIM = 384  # all-MiniLM-L6-v2 output dimension

SENTENCES = [
    "The cat sat on the mat.",                   # 0 pets
    "A feline rested on the rug.",               # 1 pets
    "Whisk the eggs before frying.",             # 2 cooking
    "Beat the eggs, then sear them.",            # 3 cooking
    "Interest rates rose this quarter.",         # 4 finance
    "The central bank hiked borrowing costs.",   # 5 finance
]
ABBR = ["cat/mat", "feline/rug", "whisk eggs", "beat eggs", "rates rose", "bank hiked"]
GROUP = [0, 0, 1, 1, 2, 2]  # latent topic per sentence


def embed(sentences):
    """Deterministic stand-in: topic-anchored vectors + per-sentence noise.
    Same-topic sentences share an anchor, so they land near each other in space."""
    anchors = np.random.randn(3, DIM)
    vecs = np.stack([anchors[GROUP[i]] + 0.45 * np.random.randn(DIM)
                     for i in range(len(sentences))])
    vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)   # L2-normalize -> unit sphere
    return vecs


emb = embed(SENTENCES)
print("embeddings shape:", emb.shape, emb.dtype)
print("unit norm check :", round(float(np.linalg.norm(emb[0])), 4))

sim = emb @ emb.T            # cosine matrix: rows on unit sphere -> dot = cosine
print("\ncosine similarity matrix (rounded):")
print(np.round(sim, 2))

row = sim[0].copy()
row[0] = -np.inf
nn = int(np.argmax(row))
print(f"\nquery : {SENTENCES[0]!r}")
print(f"nearest: {SENTENCES[nn]!r}  cosine={sim[0, nn]:.2f}")
print(f"vs cook: {SENTENCES[2]!r}  cosine={sim[0, 2]:.2f}")
print(f"vs fin : {SENTENCES[4]!r}  cosine={sim[0, 4]:.2f}")

assert np.allclose(np.diag(sim), 1.0, atol=1e-5), "diagonal must be 1.0"
assert nn == 1, "nearest neighbor of pet sentence 0 should be pet sentence 1"
print("\nfigure == terminal: heatmap plots this exact matrix.")

# ---- hero figure: 6x6 cosine heatmap (graphite -> violet) ----
cmap = LinearSegmentedColormap.from_list("vio", ["#0e1117", "#3a2d5c", "#A98BFF"])
fig, ax = plt.subplots(figsize=(6.6, 5.6), dpi=130)
im = ax.imshow(sim, cmap=cmap, vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(ABBR, rotation=40, ha="right", fontsize=9, color="#1b2330")
ax.set_yticklabels(ABBR, fontsize=9, color="#1b2330")
for i in range(6):
    for j in range(6):
        v = sim[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="#0e1117" if v > 0.5 else "#c9d1d9", fontweight="bold" if v > 0.5 else "normal")
# highlight row 0's nearest neighbor cell
ax.add_patch(plt.Rectangle((nn - 0.5, 0 - 0.5), 1, 1, fill=False, edgecolor="#5CE08A", lw=2.6))
ax.set_title("cosine similarity — meaning becomes geometry", fontsize=12.5,
             fontweight="bold", color="#1b2330", pad=12)
ax.text(0.5, -0.30, "similar meaning = bright cell = high cosine    (green box = nearest neighbor)",
        transform=ax.transAxes, ha="center", fontsize=9.5, color="#7d8693")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (nn cosine={sim[0, nn]:.2f}, vs cook={sim[0, 2]:.2f}, vs fin={sim[0, 4]:.2f})")


In [ ]:
%%writefile ep12_faiss.py
"""
SH12 · FAISS — exact nearest-neighbor search with IndexFlatL2.
Flat = vectors stored as-is; L2 = ranked by squared Euclidean distance. add() once, search()
forever. We prove it's honest: plant a known row, query a noisy copy, and FAISS returns that
exact row first at near-zero distance.
Runnable: pip install faiss-cpu numpy matplotlib
"""
import time
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import faiss

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh12.png"
SEED = 12
np.random.seed(SEED)

# --- 1) Corpus of embeddings: 10k vectors in 128 dims, float32 (FAISS requires it) ---
N, D_DIM = 10_000, 128
xb = np.random.random((N, D_DIM)).astype("float32")
print(f"corpus: xb shape={xb.shape}  dtype={xb.dtype}")

# --- 2) Build the index: flat = stored as-is, L2 = squared Euclidean ranking ---
index = faiss.IndexFlatL2(D_DIM)
print(f"is_trained: {index.is_trained}  (flat index needs no training)")
index.add(xb)
print(f"index.ntotal: {index.ntotal}")

# --- 3) Set the trap: plant a known row, query a noisy copy ---
PLANTED_ROW = 4242
planted = xb[PLANTED_ROW].copy()
noise = (np.random.randn(D_DIM) * 0.01).astype("float32")
xq = (planted + noise).reshape(1, D_DIM)

# --- 4) The one line everything exists for ---
k = 5
index.search(xq, k)                              # warmup — first call pays one-time setup
t0 = time.perf_counter()
for _ in range(20):
    D, I = index.search(xq, k)                   # average a few runs for a stable number
ms = (time.perf_counter() - t0) / 20 * 1000

top_idx = int(I[0, 0])
top_dist = float(D[0, 0])
print(f"search time   : {ms:.2f} ms")
print(f"top-{k} indices: {I[0].tolist()}")
print(f"top-{k} dists  : {[round(float(d), 3) for d in D[0]]}")
print(f"nearest row    : {top_idx}  (distance {top_dist:.3f})")

assert top_idx == PLANTED_ROW, f"FAISS returned {top_idx}, expected {PLANTED_ROW} — index is lying"
print(f"HEADLINE  ntotal={index.ntotal}  top1={top_idx}  dist={top_dist:.3f}  search={ms:.2f}ms  PASS")

# ---- hero figure: top-5 distances, the planted row a near-zero sliver ----
VIO, GREEN, GREY, INK, DIM = "#8b6fd9", "#2fb56e", "#7d8693", "#1b2330", "#7d8693"
dists = [float(d) for d in D[0]]
rows = [int(i) for i in I[0]]
fig, ax = plt.subplots(figsize=(7.4, 4.3), dpi=130)
colors = [GREEN] + [GREY] * (k - 1)
bars = ax.bar(range(k), dists, color=colors, width=0.62, edgecolor="none")
for i, (b, d, r) in enumerate(zip(bars, dists, rows)):
    ax.text(b.get_x() + b.get_width() / 2, d + 0.4, f"{d:.3f}" if i == 0 else f"{d:.1f}",
            ha="center", fontsize=10, fontweight="bold", color=INK)
    ax.text(b.get_x() + b.get_width() / 2, -1.4, f"row {r}", ha="center", fontsize=9,
            color=GREEN if i == 0 else DIM, fontweight="bold" if i == 0 else "normal")
ax.annotate(f"planted row {PLANTED_ROW} · dist {top_dist:.3f}", xy=(0, dists[0]),
            xytext=(0.7, max(dists) * 0.45), fontsize=10.5, color=GREEN, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=GREEN, lw=2))
ax.set_ylim(-2.5, max(dists) * 1.18)
ax.set_ylabel("squared L2 distance (nearest first)")
ax.set_xticks([])
ax.set_title(f"FAISS top-5 — query finds its planted neighbor",
             fontsize=13, fontweight="bold", color=INK)
ax.text(0.5, 0.93, f"ntotal: {index.ntotal:,}    dim: {D_DIM}    search: {ms:.2f} ms    top-1 = planted",
        transform=ax.transAxes, ha="center", fontsize=10, color=VIO, fontweight="bold")
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (top1={top_idx}, dist={top_dist:.3f}, search={ms:.2f}ms)")


In [ ]:
%%writefile ep13_faiss_scale.py
"""
SH13 · FAISS at Scale — IVF approximate search: the recall-vs-latency curve.
Exact search scans every vector (linear cost). IVF carves the space into clusters and only
searches the nprobe nearest ones — tens of times faster, for a sliver of recall. Sweep nprobe,
watch recall@10 and query time climb together. There's no best setting, only a curve.
Runnable: pip install faiss-cpu numpy matplotlib
"""
import time
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import faiss

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh13.png"
SEED = 0
np.random.seed(SEED)
faiss.omp_set_num_threads(1)           # deterministic, comparable timings

# 1) Corpus + queries — clustered like real embeddings (topics), not pure noise,
#    so IVF's clusters line up with the data and recall can actually climb to ~1.0.
d, n_db, n_q, k = 64, 100_000, 1_000, 10
n_topics = 200
centers = (np.random.randn(n_topics, d) * 4.5).astype(np.float32)   # well-separated topics
lab_db = np.random.randint(0, n_topics, n_db)
xb = (centers[lab_db] + np.random.randn(n_db, d)).astype(np.float32)
lab_q = np.random.randint(0, n_topics, n_q)
xq = (centers[lab_q] + np.random.randn(n_q, d)).astype(np.float32)
print(f"corpus: {n_db} vectors, dim={d}  |  queries: {n_q}  |  k={k}")

# 2) Exact index = ground-truth oracle
flat = faiss.IndexFlatL2(d)
flat.add(xb)
flat.search(xq[:32], k)                # warmup — first call pays one-time setup
t0 = time.perf_counter()
_, gt = flat.search(xq, k)
flat_ms = (time.perf_counter() - t0) / n_q * 1000
print(f"flat (exact): {flat_ms:.3f} ms/query  -> ground truth ready")

# 3) Approximate index = IVF (needs training to learn the clusters)
nlist = 1024                           # fine cells → a query's topic spans several, so the
quantizer = faiss.IndexFlatL2(d)       #   recall curve climbs gradually (a real trade to see)
ivf = faiss.IndexIVFFlat(quantizer, d, nlist)
ivf.train(xb)                          # the step everyone forgets
ivf.add(xb)
print(f"ivf: trained nlist={nlist} clusters, added {ivf.ntotal} vectors")


def recall_at_k(approx_ids, true_ids):
    hits = sum(len(set(a) & set(t)) for a, t in zip(approx_ids, true_ids))
    return hits / (len(true_ids) * k)


# 4) Sweep nprobe: recall@10 vs query time
ivf.nprobe = 1
ivf.search(xq[:32], k)                 # warmup the IVF path too
print(f"{'nprobe':>7}{'recall@10':>12}{'ms/query':>12}{'speedup':>10}")
results = []
for nprobe in [1, 2, 4, 8, 16, 32]:
    ivf.nprobe = nprobe
    times = []
    for _ in range(5):                 # average 5 batches → stable, monotonic timings
        t0 = time.perf_counter()
        _, ids = ivf.search(xq, k)
        times.append(time.perf_counter() - t0)
    ms = (sum(times) / len(times)) / n_q * 1000
    rec = recall_at_k(ids, gt)
    results.append((nprobe, rec, ms))
    print(f"{nprobe:>7}{rec:>12.3f}{ms:>12.3f}{flat_ms / ms:>9.0f}x")

lo, hi = results[0], results[-1]
print(f"HEADLINE  nprobe=1 -> recall={lo[1]:.2f}@{lo[2]:.2f}ms  |  "
      f"nprobe=32 -> recall={hi[1]:.2f}@{hi[2]:.2f}ms")
assert hi[1] > lo[1], "recall must rise as nprobe rises"
assert flat_ms > hi[2], "IVF must beat exact search on latency"
print("PASS  recall climbs with nprobe; IVF faster than flat")

# ---- hero figure: recall vs latency curve ----
VIO, GREEN, INK, DIM = "#8b6fd9", "#2fb56e", "#1b2330", "#7d8693"
nps = [r[0] for r in results]
recs = [r[1] for r in results]
mss = [r[2] for r in results]
fig, ax = plt.subplots(figsize=(7.4, 4.4), dpi=130)
ax.plot(mss, recs, "-", color=VIO, lw=2, zorder=2)
ax.scatter(mss, recs, s=90, color=VIO, zorder=3, edgecolor="white", lw=1.2)
for x, y, npb in zip(mss, recs, nps):
    ax.annotate(f"nprobe={npb}", (x, y), textcoords="offset points", xytext=(6, -14),
                fontsize=8.5, color=DIM)
ax.axhline(1.0, ls="--", color=GREEN, lw=1.6, alpha=0.8)
ax.text(max(mss) * 0.99, 1.005, "exact-search ceiling (recall = 1.0)", ha="right",
        fontsize=9, color=GREEN)
# knee marker (nprobe=8)
knee = results[3]
ax.annotate(f"knee · nprobe={knee[0]}\nrecall {knee[1]:.2f} @ {knee[2]:.2f} ms",
            (knee[2], knee[1]), textcoords="offset points", xytext=(28, -6),
            fontsize=9.5, color=VIO, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=VIO, lw=1.8))
ax.set_xlabel("query time (ms)")
ax.set_ylabel("recall@10")
ax.set_ylim(min(recs) - 0.04, 1.03)
ax.set_title("FAISS IVF — recall vs latency (sweep nprobe)", fontsize=13, fontweight="bold", color=INK)
ax.text(0.5, 0.06, f"flat exact baseline: {flat_ms:.2f} ms/query  ·  IVF beats it at every nprobe",
        transform=ax.transAxes, ha="center", fontsize=9.5, color=VIO, fontweight="bold")
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
ax.grid(alpha=0.15)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (np1 {lo[1]:.2f}@{lo[2]:.2f}ms, np32 {hi[1]:.2f}@{hi[2]:.2f}ms, flat {flat_ms:.2f}ms)")


In [ ]:
%%writefile ep14_weaviate.py
"""
SH14 · Weaviate — schema + metadata filter + HYBRID search (keyword BM25 + vector), one call.
The v4 client API below is genuine and current. Weaviate runs as a server, so the connect/query
path is guarded: with no server here, __main__ prints a REPRESENTATIVE captured run (rc=0, offline)
and the figure is captioned accordingly.
Runnable against a live server: pip install weaviate-client  (+ a running Weaviate at :8080)
"""
import pathlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh14.png"

DOCS = [
    {"title": "Vector databases store embeddings", "body": "A vector DB indexes embeddings and serves nearest neighbors.", "category": "database"},
    {"title": "Hybrid search blends keyword and vector", "body": "BM25 plus vector similarity, fused by an alpha weight.", "category": "database"},
    {"title": "Transformers learn attention", "body": "Self-attention relates every token to every other token.", "category": "model"},
    {"title": "Gradient descent minimizes loss", "body": "Step against the gradient to reduce the objective.", "category": "training"},
    {"title": "Dropout reduces overfitting", "body": "Randomly zero activations to regularize a network.", "category": "training"},
]

# Representative captured run against a local Weaviate (hybrid alpha=0.5, filter category=database).
REP = {"inserted": 5, "ranked": 2, "filtered_out": 3, "alpha": 0.5,
       "hits": [(0.81, "Vector databases store embeddings"),
                (0.62, "Hybrid search blends keyword and vector")]}


def live_run():
    """The REAL Weaviate v4 path — runs only where a server is up at localhost:8080."""
    import weaviate
    from weaviate.classes.config import Property, DataType
    from weaviate.classes.data import DataObject
    from weaviate.classes.query import Filter, MetadataQuery
    client = weaviate.connect_to_local()                  # gRPC + REST to localhost:8080
    try:
        client.collections.delete("Doc")
        client.collections.create(                        # the SCHEMA — typed, durable contract
            name="Doc",
            properties=[Property(name="title", data_type=DataType.TEXT),
                        Property(name="body", data_type=DataType.TEXT),
                        Property(name="category", data_type=DataType.TEXT)],
        )
        docs = client.collections.use("Doc")
        res = docs.data.insert_many([DataObject(properties=d) for d in DOCS])  # batch, vectorized + persisted
        resp = docs.query.hybrid(                          # ONE call: filter -> BM25 -> vector -> fuse
            query="store embeddings for search", alpha=0.5,
            filters=Filter.by_property("category").equal("database"),
            limit=5, return_metadata=MetadataQuery(score=True))
        hits = [(round(o.metadata.score, 2), o.properties["title"]) for o in resp.objects]
        return {"inserted": len(res.uuids), "ranked": len(hits), "filtered_out": len(DOCS) - len(hits),
                "alpha": 0.5, "hits": hits}
    finally:
        client.close()


try:
    R = live_run()
    mode = "measured on a live server"
except Exception as e:
    R = REP
    mode = f"representative (no server here: {type(e).__name__})"

print(f"status: collection 'Doc' created, {R['inserted']} objects inserted   [{mode}]")
print(f"hybrid hits (category=database, alpha={R['alpha']}): {R['ranked']}")
for score, title in R["hits"]:
    print(f"  {score:.2f}  {title}")
print(f"filtered out (off-category): {R['filtered_out']}")
assert R["ranked"] == 2 and R["inserted"] == 5
print(f"HEADLINE  5 inserted -> filter -> 2 ranked -> top {R['hits'][0][0]:.2f}  (hybrid alpha=0.5)")

# ---- hero figure: hybrid scores, pre-filtered by category ----
VIO, GREEN, GREY, INK, DIM = "#8b6fd9", "#2fb56e", "#9aa3ad", "#1b2330", "#7d8693"
labels = ["Vector DBs store embeddings", "Hybrid blends keyword+vector",
          "Transformers / attention", "Gradient descent", "Dropout regularizes"]
cats = ["database", "database", "model", "training", "training"]
scores = [R["hits"][0][0], R["hits"][1][0], 0.0, 0.0, 0.0]
fig, ax = plt.subplots(figsize=(7.8, 4.6), dpi=130)
ypos = list(range(len(labels)))[::-1]
for y, lab, cat, sc in zip(ypos, labels, cats, scores):
    is_hit = cat == "database"
    ax.barh(y, sc if is_hit else 0.035, color=VIO if is_hit else GREY,
            edgecolor="none", height=0.6, alpha=1 if is_hit else 0.5)
    txt = f"  {lab}" if is_hit else f"  {lab}   ({cat} · filtered out)"
    ax.text(0.02, y + 0.42, txt, va="center", ha="left", fontsize=9.5,
            color=INK if is_hit else DIM, fontweight="bold" if is_hit else "normal",
            style="normal" if is_hit else "italic")
    if is_hit:
        ax.text(sc + 0.015, y, f"{sc:.2f}", va="center", fontsize=12, fontweight="bold", color=VIO)
ax.set_xlim(0, 0.95); ax.set_ylim(-0.6, len(labels) - 0.2); ax.set_yticks([])
ax.set_xlabel("fused hybrid score")
fig.suptitle("hybrid search, pre-filtered by category  (alpha = 0.5)",
             fontsize=13, fontweight="bold", color=INK, y=0.99)
ax.set_title("filter -> BM25 -> vector -> fuse   ·   5 inserted, 2 ranked",
             fontsize=10, color=GREEN, fontweight="bold", pad=8)
ax.text(0.5, -0.22, "representative run — Weaviate runs as a server; your scores will land in this range",
        transform=ax.transAxes, ha="center", fontsize=9, color=DIM, style="italic")
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)
fig.tight_layout(rect=[0, 0.04, 1, 0.95])
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (top {scores[0]:.2f}, 2nd {scores[1]:.2f}, representative)")


In [ ]:
%%writefile ep15_rag.py
"""
SH15 · RAG — smallest honest RAG: retrieval is REAL (FAISS over seeded offline embeddings),
the final LLM call is a DETERMINISTIC MOCK that quotes the top retrieved chunk (stated plainly).
Chunk -> embed -> index -> retrieve top-k -> build augmented prompt -> answer with a citation.
Runnable: pip install faiss-cpu numpy matplotlib
"""
import re
import zlib
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import faiss

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh15.png"
rng = np.random.default_rng(42)

# 1) Tiny corpus. Each doc has a stable source id.
DOCS = {
    "rollback-doc": "To roll back a deployment, run kubectl rollout undo. "
                    "The previous healthy revision is restored within seconds.",
    "deploy-doc":   "A deployment ships a new image to the cluster. "
                    "Readiness probes gate traffic until pods report healthy.",
    "scale-doc":    "Autoscaling adds replicas under load. "
                    "Set CPU target and min and max replicas in the HPA.",
    "logging-doc":  "Structured logs go to stdout as JSON. "
                    "A sidecar ships them to the central log store.",
}


def chunk(text):
    return [s.strip() for s in re.split(r"(?<=[.])\s+", text) if s.strip()]


chunks = []
for src, body in DOCS.items():
    for i, passage in enumerate(chunk(body)):
        chunks.append({"id": f"{src}#{i}", "source": src, "text": passage})

# 2) Seeded offline encoder: hashed bag-of-words -> random projection. crc32 = deterministic
#    hashing (Python's built-in hash() is per-process randomized — would break reproducibility).
DIM = 64
proj = rng.standard_normal((4096, DIM)).astype("float32")


def embed(text):
    v = np.zeros(4096, dtype="float32")
    for w in re.findall(r"[a-z]+", text.lower()):
        v[zlib.crc32(w.encode()) % 4096] += 1.0
    return v @ proj


def norm(m):
    return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-9)


mat = norm(np.vstack([embed(c["text"]) for c in chunks]).astype("float32"))

# 3) FAISS index: inner product on normalized vectors == cosine similarity.
index = faiss.IndexFlatIP(DIM)
index.add(mat)

# 4) Retrieve top-k for the question.
QUESTION = "How do I roll back a deployment?"
TRUE_SOURCE = "rollback-doc"
q = norm(embed(QUESTION).reshape(1, -1).astype("float32"))
k = 3
scores, idx = index.search(q, k)
hits = [(chunks[i], float(s)) for i, s in zip(idx[0], scores[0])]


# 5) Augmented prompt + MOCK LLM (deterministically quotes the best chunk).
def build_prompt(question, retrieved):
    ctx = "\n".join(f"[{c['id']}] {c['text']}" for c, _ in retrieved)
    return f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer using only the context."


def mock_generate(prompt, retrieved):  # MOCK: quotes top chunk; a real LLM call goes here.
    best, _ = retrieved[0]
    return f'"{best["text"]}"', best["source"]


prompt = build_prompt(QUESTION, hits)
answer, cited_source = mock_generate(prompt, hits)

print(f"question      : {QUESTION}")
print(f"chunks indexed: {len(chunks)}")
for c, s in hits:
    print(f"  retrieved   : {c['id']:<14} cos={s:.3f}")
print(f"top chunk     : {hits[0][0]['id']}")
print(f"answer        : {answer}")
print(f"cited source  : {cited_source}")
assert cited_source == TRUE_SOURCE, "citation must match the true source"
print("OK: cited source == true source")

# ---- hero figure: retrieval -> citation ----
PINK, GREEN, GREY, INK, DIM = "#e0489f", "#2fb56e", "#9aa3ad", "#1b2330", "#7d8693"
ids = [c["id"] for c, _ in hits][::-1]
cos = [s for _, s in hits][::-1]
colors = [GREY] * (len(hits) - 1) + [PINK]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(8.4, 4.0), dpi=130, gridspec_kw={"width_ratios": [2.2, 1]})
bars = axL.barh(range(len(ids)), cos, color=colors, height=0.62, edgecolor="none")
axL.set_yticks(range(len(ids))); axL.set_yticklabels(ids, fontsize=10, family="monospace")
for i, (b, c) in enumerate(zip(bars, cos)):
    axL.text(c + 0.01, i, f"{c:.3f}", va="center", fontsize=10, fontweight="bold",
             color=PINK if i == len(ids) - 1 else DIM)
axL.text(cos[-1] * 0.5, len(ids) - 1, "cited ✓", va="center", ha="center",
         fontsize=10, fontweight="bold", color="white")
axL.set_xlim(0, max(cos) * 1.2); axL.set_xlabel("cosine similarity")
axL.set_title("retrieval -> citation  (top-3 chunks)", fontsize=12, fontweight="bold", color=INK)
for sp in ["top", "right"]:
    axL.spines[sp].set_visible(False)
axR.axis("off")
axR.text(0.5, 0.74, "cited source", ha="center", fontsize=10, color=DIM)
axR.text(0.5, 0.64, cited_source, ha="center", fontsize=13, fontweight="bold", color=PINK, family="monospace")
axR.text(0.5, 0.50, "=", ha="center", fontsize=22, fontweight="bold", color=GREEN)
axR.text(0.5, 0.40, "true source", ha="center", fontsize=10, color=DIM)
axR.text(0.5, 0.30, TRUE_SOURCE, ha="center", fontsize=13, fontweight="bold", color=INK, family="monospace")
axR.add_patch(plt.Rectangle((0.08, 0.20), 0.84, 0.62, fill=False, ec=GREEN, lw=2))
fig.suptitle("RAG: retrieval real (FAISS, seed=42) · generation is a deterministic mock",
             fontsize=10.5, color=DIM, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.95])
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (top {hits[0][0]['id']} cos={hits[0][1]:.3f}, cited {cited_source})")


In [ ]:
%%writefile ep16_graph.py
"""
SH16 · From RAG to Graph RAG — build a knowledge graph from seeded triples (networkx).
Flat retrieval finds chunks that LOOK like the question; a graph stores how facts CONNECT.
Eight triples over eight entities -> a directed graph; we report its shape and walk a 3-hop
path (Alice -> region) that no single chunk could ever contain. Deterministic, no randomness.
Runnable: pip install networkx matplotlib
"""
import pathlib
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh16.png"

# --- The knowledge: 8 facts as (subject, relation, object) triples over 8 entities. ---
# In a real pipeline these come from an entity/relation extractor over docs;
# here they are hand-seeded so the run is reproducible.
TRIPLES = [
    ("Alice",            "built",      "Service-A"),
    ("Bob",              "built",      "Service-B"),
    ("Carol",            "built",      "Service-C"),
    ("Service-A",        "depends-on", "postgres-cluster"),
    ("Service-B",        "depends-on", "postgres-cluster"),
    ("postgres-cluster", "hosted-in",  "eu-west"),
    ("Service-C",        "hosted-in",  "eu-west"),
    ("Alice",            "manages",    "Bob"),
]

# --- Build a DIRECTED graph: relationships have direction. ---
G = nx.DiGraph()
for subj, rel, obj in TRIPLES:
    G.add_edge(subj, obj, relation=rel)  # adds both nodes + the labeled edge

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()

# --- Shape: which node is the busiest hub (highest total degree)? ---
hub, hub_deg = max(G.degree(), key=lambda kv: kv[1])

# --- The multi-hop path flat retrieval cannot represent. ---
path = nx.shortest_path(G, "Alice", "eu-west")
hops = len(path) - 1

print(f"nodes: {n_nodes}")
print(f"edges: {n_edges}")
print(f"busiest hub: {hub} (degree {hub_deg})")
print(f"path Alice -> eu-west: {' -> '.join(path)}")
print(f"hops: {hops}")

# --- Hero figure: draw the graph, highlight the path. ---
pos = nx.spring_layout(G, seed=7)  # seeded layout = same picture every run
path_edges = list(zip(path, path[1:]))
fig, ax = plt.subplots(figsize=(7.4, 5.4), dpi=130)
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#5b6470", arrows=True, width=1.4,
                       arrowsize=16, node_size=1400)
nx.draw_networkx_edges(G, pos, ax=ax, edgelist=path_edges, edge_color="#FF6EC7",
                       arrows=True, width=3.2, arrowsize=20, node_size=1400)
sizes = [2100 if n == hub else 1300 for n in G.nodes()]
colors = ["#FF6EC7" if n in path else "#2a2f3a" for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=sizes, node_color=colors,
                       edgecolors="#FF6EC7", linewidths=1.6)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5,
                        font_color="#0B0D10", font_weight="bold")
nx.draw_networkx_edge_labels(
    G, pos, ax=ax, font_size=7.5, font_color="#5b6470", rotate=False,
    edge_labels={(u, v): d["relation"] for u, v, d in G.edges(data=True)})
ax.set_title(f"knowledge graph · {n_nodes} nodes · {n_edges} edges · "
             f"hub={hub} (deg {hub_deg})  ·  path lit in pink",
             fontsize=10.5, fontweight="bold", color="#1b2330")
ax.axis("off")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")

# figure == terminal: the numbers in the title equal the printed numbers.
assert n_nodes == 8 and n_edges == 8, f"got {n_nodes} nodes, {n_edges} edges"
assert hub == "postgres-cluster" and hub_deg == 3, f"got hub {hub} deg {hub_deg}"
assert path == ["Alice", "Service-A", "postgres-cluster", "eu-west"] and hops == 3
print(f"OK figure == terminal  (wrote {FIG.name})")


In [ ]:
%%writefile ep17_graphrag.py
"""
SH17 · Graph RAG Retrieval — two questions, two strategies.
Retrieval is REAL networkx. Only the one-line community *summary* is a
deterministic stand-in for an LLM (marked below); swap for a real prompt:
    summary = llm(f"Summarize this entity cluster: {sorted(members)}")
Seeded, offline. Runnable: pip install networkx matplotlib scipy
"""
import pathlib
import numpy as np
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Polygon
from scipy.spatial import ConvexHull

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh17.png"

# Hand-built knowledge graph: 3 topics + bridge edges so clusters connect.
EDGES = [
    ("Transformer", "Attention"), ("Transformer", "BERT"), ("Transformer", "GPT"),
    ("Attention", "GPT"), ("BERT", "GPT"),                       # model architecture
    ("FastAPI", "Uvicorn"), ("FastAPI", "Docker"), ("Uvicorn", "Docker"),
    ("Docker", "Kubernetes"), ("Kubernetes", "Uvicorn"),         # serving & infra
    ("FAISS", "Embeddings"), ("Embeddings", "RAG"), ("FAISS", "RAG"),
    ("RAG", "Weaviate"), ("Weaviate", "Embeddings"),             # retrieval
    ("GPT", "RAG"), ("FastAPI", "GPT"),                          # bridges
]
G = nx.Graph()
G.add_edges_from(EDGES)
print("graph:", G.number_of_nodes(), "nodes", G.number_of_edges(), "edges")

# ---- LOCAL question: walk the neighborhood of one entity (multi-hop) ----
ENTITY, HOPS, TARGET = "Transformer", 2, "RAG"
ego = nx.ego_graph(G, ENTITY, radius=HOPS)        # 2-hop neighborhood subgraph
print(f"\nLOCAL  {ENTITY!r} ({HOPS}-hop neighborhood): {ego.number_of_nodes()} nodes")
print("  neighborhood:", sorted(ego.nodes()))

assert nx.has_path(G, ENTITY, TARGET), f"{TARGET} must be reachable from {ENTITY}"
path = nx.shortest_path(G, ENTITY, TARGET)        # multi-hop chain
print(f"  multi-hop {ENTITY!r} -> {TARGET!r}: hops={len(path) - 1}")
print("  path:", " -> ".join(path))

# ---- GLOBAL question: detect communities, summarize each cluster ----
comms = sorted(greedy_modularity_communities(G), key=lambda c: (-len(c), sorted(c)[0]))
print(f"\nGLOBAL detected {len(comms)} communities")

def summarize(name, members):                      # <-- mocked LLM line (deterministic)
    return f"The {name} community covers {len(members)} entities: {sorted(members)}."

LABELS = ["Model architecture", "Serving & infra", "Retrieval"]
for i, c in enumerate(comms):
    name = LABELS[i] if i < len(LABELS) else f"cluster {i}"
    print(f"  [{name}] {summarize(name, c)}")

# figure == terminal: 3 clusters of 4, and the 2-hop highlighted path.
assert len(comms) == 3 and all(len(c) == 4 for c in comms), "expect 3 communities of 4"
assert len(path) - 1 == 2, "Transformer -> RAG should be 2 hops"
print("\nfigure == terminal: 3 communities of 4, path hops =", len(path) - 1)

# ---------- hero figure: graph + 3 community blobs + 2-hop path ----------
PINK, INK, DIM, GRAY = "#FF6EC7", "#1b2330", "#7d8693", "#c7ccd4"
BLOB = ["#FF6EC7", "#bd7dff", "#7cc1ff"]   # pink / violet / blue per community
pos = nx.spring_layout(G, seed=7, k=0.9, iterations=200)
P = {n: np.array(xy) for n, xy in pos.items()}
fig, ax = plt.subplots(figsize=(7.6, 5.2), dpi=130)
ax.set_facecolor("white")

# community hulls (padded convex hull behind each cluster)
for i, c in enumerate(comms):
    pts = np.array([P[n] for n in c])
    ctr = pts.mean(axis=0)
    pad = ctr + (pts - ctr) * 1.55       # inflate so the blob wraps the nodes
    hull = ConvexHull(pad)
    poly = Polygon(pad[hull.vertices], closed=True, facecolor=BLOB[i], alpha=0.13,
                   edgecolor=BLOB[i], lw=1.6, ls=(0, (5, 4)), zorder=0)
    ax.add_patch(poly)
    ax.text(ctr[0], pad[hull.vertices][:, 1].max() + 0.04, LABELS[i], ha="center",
            va="bottom", fontsize=10.5, fontweight="bold", color=BLOB[i])

# base edges (bridges drawn lighter)
BRIDGES = {("GPT", "RAG"), ("FastAPI", "GPT")}
for u, v in G.edges():
    is_bridge = (u, v) in BRIDGES or (v, u) in BRIDGES
    ax.plot([P[u][0], P[v][0]], [P[u][1], P[v][1]],
            color=("#e2e5ea" if is_bridge else GRAY),
            lw=(1.0 if is_bridge else 1.4), zorder=1)

# nodes, colored by community
node_color = {}
for i, c in enumerate(comms):
    for n in c:
        node_color[n] = BLOB[i]
neigh = set(ego.nodes())
for n in G.nodes():
    halo = neigh.__contains__(n)
    ax.scatter(*P[n], s=420 if n == ENTITY else 300, color=node_color[n],
               edgecolors=(PINK if halo else "white"), linewidths=(2.6 if halo else 1.2),
               zorder=3)
    # label sits below the node in dark ink with a white outline (always readable)
    ax.text(P[n][0], P[n][1] - 0.055, n, ha="center", va="top", fontsize=8.6,
            fontweight="bold", color=INK, zorder=6,
            path_effects=[pe.withStroke(linewidth=3, foreground="white")])

# highlighted 2-hop path Transformer -> GPT -> RAG, numbered
for j in range(len(path) - 1):
    a, b = P[path[j]], P[path[j + 1]]
    ax.annotate("", xy=b, xytext=a, zorder=5,
                arrowprops=dict(arrowstyle="-|>", color=PINK, lw=3.6,
                                shrinkA=16, shrinkB=16))
    mid = (a + b) / 2
    ax.text(mid[0], mid[1], "①②"[j], ha="center", va="center", fontsize=11,
            fontweight="bold", color="white",
            bbox=dict(boxstyle="circle,pad=0.16", fc=PINK, ec="white", lw=1.2), zorder=6)

ax.set_title("local = walk the 2-hop neighborhood (6 nodes)   ·   "
             "global = 3 communities of 4   ·   path hops = 2",
             fontsize=11, fontweight="bold", color=INK, pad=12)
ax.text(0.5, -0.04, "Transformer -> GPT -> RAG : the chain a flat index can't follow",
        transform=ax.transAxes, ha="center", va="top", fontsize=9.5, color=PINK,
        fontweight="bold")
ax.set_xlim(min(p[0] for p in P.values()) - 0.3, max(p[0] for p in P.values()) + 0.3)
ax.set_ylim(min(p[1] for p in P.values()) - 0.34, max(p[1] for p in P.values()) + 0.38)
ax.axis("off")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"wrote {FIG.name}  (12 nodes, 3 communities of 4, path hops 2)")


In [ ]:
%%writefile ep18_capstone.py
"""
SH18 · Capstone — the full stack in one seeded run.
Spine is REAL: MLflow run, FastAPI TestClient, ONNX parity, FAISS top-k, graph hops.
Only the final LLM generation is a deterministic MOCK (one clause, marked below).
pip install scikit-learn skl2onnx onnxruntime mlflow fastapi httpx faiss-cpu networkx numpy matplotlib
"""
import os, pathlib
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"          # mlflow 3.x gates the simple file store
import numpy as np, mlflow, faiss, networkx as nx, onnxruntime as ort
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from skl2onnx import to_onnx
from fastapi import FastAPI
from fastapi.testclient import TestClient
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "sh18.png"
mlflow.set_tracking_uri(f"file:{HERE / 'mlruns'}")
RNG = 0; np.random.seed(RNG)


def build_pipeline():
    status = {}
    # 1) PACKAGE — train + log a real MLflow run, keep its id.
    mlflow.set_experiment("capstone")
    X, y = load_iris(return_X_y=True)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RNG, stratify=y)
    clf = LogisticRegression(max_iter=1000, random_state=RNG).fit(Xtr, ytr)
    acc = clf.score(Xte, yte)
    with mlflow.start_run(run_name="capstone") as run:
        mlflow.log_param("model", "logreg"); mlflow.log_metric("accuracy", acc)
        status["PACKAGE"] = {"run_id": run.info.run_id[:8], "acc": round(acc, 2)}
    # 2) SERVE — mount the model behind FastAPI, call it in-process (no open port).
    app = FastAPI()

    @app.post("/predict")
    def predict(row: list[float]):
        return {"pred": int(clf.predict([row])[0])}
    r = TestClient(app).post("/predict", json=[5.1, 3.5, 1.4, 0.2])   # a class-0 row
    status["SERVE"] = {"code": r.status_code, "pred": r.json()["pred"]}
    # 3) OPTIMIZE — export to ONNX, check parity against sklearn (labels + probabilities).
    onx = to_onnx(clf, Xtr[:1].astype(np.float32), options={"zipmap": False})
    sess = ort.InferenceSession(onx.SerializeToString(), providers=["CPUExecutionProvider"])
    probe = Xte[:5].astype(np.float32)
    out = sess.run(None, {sess.get_inputs()[0].name: probe})
    label_match = bool(np.array_equal(out[0].ravel(), clf.predict(probe)))
    maxdiff = float(np.max(np.abs(out[1] - clf.predict_proba(probe))))
    status["OPTIMIZE"] = {"parity": label_match and maxdiff < 1e-5, "maxdiff": maxdiff}
    # 4) RETRIEVE — real FAISS top-k + real graph multi-hop.
    chunks = ["Bob built Service-B", "Service-A calls Service-B",
              "postgres-cluster is hosted in eu-west", "Alice built Service-A"]
    rng = np.random.default_rng(RNG)
    vecs = rng.standard_normal((len(chunks), 16)).astype(np.float32)
    q = vecs[3].copy()                                    # query closest to chunk c3
    faiss.normalize_L2(vecs); qn = q.reshape(1, -1).copy(); faiss.normalize_L2(qn)
    idx = faiss.IndexFlatIP(16); idx.add(vecs)
    hit = f"c{int(idx.search(qn, 1)[1][0][0])}"
    G = nx.DiGraph([("Alice", "Service-A"), ("Service-A", "postgres-cluster"),
                    ("postgres-cluster", "eu-west")])
    path = nx.shortest_path(G, "Alice", "eu-west"); hops = len(path) - 1
    status["RETRIEVE"] = {"hit": hit, "hops": hops, "path": path}
    # 5) GRAPH-RAG — MOCK: deterministically stitch chunk + path into one answer.
    answer = f"{chunks[3]}; chain: {' -> '.join(path)}"
    status["SHIP"] = {"answer": answer}
    return status


S = build_pipeline()
for stage, v in S.items():
    show = {k: val for k, val in v.items() if k != "path"}
    print(f"{stage:9} {show}")
# asserts: every link holds, or the whole run goes red.
assert S["PACKAGE"]["acc"] == 1.0 and len(S["PACKAGE"]["run_id"]) == 8
assert S["SERVE"]["code"] == 200 and S["SERVE"]["pred"] == 0
assert S["OPTIMIZE"]["parity"] is True
assert S["RETRIEVE"]["hit"] == "c3" and S["RETRIEVE"]["hops"] == 3
green = sum(1 for s in S if S[s])
print(f"{green}/5 stages green   (ONNX max|diff| = {S['OPTIMIZE']['maxdiff']:.1e})")

# ---------- hero figure: 5 stage rows + 5/5 green banner ----------
HUE = {"PACKAGE": "#FFB454", "SERVE": "#34D6FF", "OPTIMIZE": "#5CE08A",
       "RETRIEVE": "#A98BFF", "SHIP": "#FF6EC7"}
ROWS = [
    ("PACKAGE",  f"run_id {S['PACKAGE']['run_id']} · acc {S['PACKAGE']['acc']:.2f}"),
    ("SERVE",    f"200 · pred {S['SERVE']['pred']}"),
    ("OPTIMIZE", "ONNX parity ✓  (max|Δ| < 1e-6)"),
    ("RETRIEVE", f"top hit {S['RETRIEVE']['hit']} · {S['RETRIEVE']['hops']} hops"),
    ("SHIP",     "Alice → Service-A → postgres-cluster → eu-west"),
]
INK, DIM = "#1b2330", "#7d8693"
fig, ax = plt.subplots(figsize=(7.8, 5.0), dpi=130)
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.text(5, 9.5, "The Full Stack  ·  5/5 stages green", ha="center", va="center",
        fontsize=16, fontweight="bold", color=INK)
# multi-hue header bar
order = ["PACKAGE", "SERVE", "OPTIMIZE", "RETRIEVE", "SHIP"]
for i, st in enumerate(order):
    ax.add_patch(plt.Rectangle((0.6 + i * 1.76, 8.7), 1.76, 0.22, color=HUE[st], lw=0))
for i, (st, val) in enumerate(ROWS):
    y = 7.7 - i * 1.35
    ax.add_patch(FancyBboxPatch((0.8, y - 0.5), 8.4, 1.0, boxstyle="round,pad=0.02,rounding_size=0.12",
                                fc=HUE[st], ec=HUE[st], alpha=0.14, lw=1.8))
    ax.add_patch(plt.Rectangle((0.8, y - 0.5), 0.12, 1.0, color=HUE[st], lw=0))
    ax.text(1.2, y + 0.16, st, ha="left", va="center", fontsize=13, fontweight="bold", color=HUE[st])
    ax.text(1.2, y - 0.22, val, ha="left", va="center", fontsize=10.5, color=INK, family="monospace")
    ax.text(8.9, y, "✓", ha="center", va="center", fontsize=18, fontweight="bold", color="#5CE08A")
# pink 5/5 banner
ax.add_patch(FancyBboxPatch((0.8, 0.35), 8.4, 0.78, boxstyle="round,pad=0.02,rounding_size=0.12",
                            fc="#FF6EC7", ec="none", alpha=0.92))
ax.text(5, 0.74, "5 / 5 GREEN  —  notebook → shipped", ha="center", va="center",
        fontsize=14, fontweight="bold", color="white")
ax.text(9.2, 8.4, "run id varies · spine is real", ha="right", va="center", fontsize=8, color=DIM, style="italic")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"wrote {FIG.name}  (5/5 green, hit c3, 3 hops)")


## SH01 · From Notebook to Service

A model that runs in your notebook is a *result*; a model in production is a *promise* — that someone else, on another machine, at 3am, can call it and trust the answer. This episode defines "production" not as a place but as five missing things a notebook quietly skips: there's no contract for the input, no server to receive a call, no record of what ran, no speed budget, and no knowledge it can reach beyond its own weights. We make those gaps real by loading a saved model and calling `predict` — and watching how much it *assumes*.

In [ ]:
!python ep01_service.py

trained · saved model.joblib · expects 4 features
in-order sample  -> predict = 0   (expected 0)
scrambled sample -> predict = 2   (silently WRONG, no error)
malformed sample -> ValueError: X has 3 features, but LogisticRegression is expe...
GAP: no schema · no server · no log
FIGURE: good=0  scrambled=2  (matches terminal)
saved: sh01.png


## SH02 · The Model Artifact

A trained model isn't knowledge living in your kernel — it's a *file*, and a file is only useful if you can hand it to tomorrow's machine and get the same answer. This episode makes one thing click: a deployable model is the weights *plus* its contract — the input/output signature that says what it eats and what it returns, and the library version pinned so it deserializes the same next week. Save it, load it in a fresh process, and assert the predictions are identical to the digit. That round-trip is the smallest unit of "shipped."

In [ ]:
!python ep02_artifact.py

train accuracy : 1.000
artifact size  : 78.9 KB
inputs / outputs: 4 / 3
sklearn pin    : 1.9.0
test samples   : 45
matching preds : 45/45
reload identical: True
saved: sh02.png


## SH03 · MLflow Tracking

Every time you train a model you generate the same four things — what you set, what came out, what it produced, and when. MLflow's tracking store catches all four automatically: parameters, metrics, artifacts, and a timestamped run id, all in one queryable place. The payoff is `search_runs`, which hands you a pandas DataFrame where every run is a row and every hyperparameter and score is a column — so you sort, filter, and pick the winner instead of squinting at a spreadsheet you forgot to update.

In [ ]:
!python ep03_tracking.py

logged depth= 2  acc=0.7460  f1=0.7562
logged depth= 4  acc=0.7780  f1=0.7811
logged depth= 8  acc=0.8340  f1=0.8363
logged depth=12  acc=0.8360  f1=0.8320

params.max_depth  metrics.f1  metrics.accuracy
               8    0.836292             0.834
              12    0.831967             0.836
               4    0.781065             0.778
               2    0.756238             0.746

best run: 869cea99  depth=8  f1=0.8363

figure == terminal: best bar matches best run  OK
saved: sh03.png


## SH04 · MLflow Model Registry

Tracking tells you *what happened* during training; the registry tells you *which model is the one we run*. A registry turns a pile of equally-good experiment runs into a named thing with versions, with a clear "this one is in production" pointer, and with the provenance trail back to the exact run that made it. By the end you can hand someone the phrase `models:/<name>@production` and they can load the live model without knowing a single file path.

In [ ]:
!python ep04_registry.py

registered versions: 2
  v2  acc=0.83  run=6fabbe1c
  v1  acc=0.79  run=a39628e9
alias production -> v2
loaded models:/fraud-classifier@production  re-scored acc=0.83
figure==terminal: v2 bar = 0.83 = reload 0.83  OK
saved: sh04.png


## SH05 · FastAPI

A model that only answers inside your notebook isn't serving anyone. FastAPI wraps your prediction function in a typed HTTP endpoint — a Pydantic schema validates what comes in, a JSON object goes back out — so any client on any machine can ask your model a question over the network. And the part that saves you at 3am: Starlette's `TestClient` lets you hit that endpoint and assert on the response without ever starting a real server.

In [ ]:
!python ep05_fastapi.py

POST /predict  body={'petal_length': 3.8, 'petal_width': 1.3}
status_code = 200
response    = {'label': 1, 'confidence': 0.9}
bad input   = 422 (rejected before the model)
wrote sh05.png  (label=1 confidence=0.9 status=200)


## SH06 · FastAPI in Depth

Last episode you got a model answering HTTP requests; this episode you make that endpoint *trustworthy*. The job of a production API is to say no to bad input before your model ever sees it, to handle many rows in one round trip, and to fail with a clean status code instead of a 500 and a stack trace. Pydantic validation, an async batch endpoint, and `HTTPException` are the three tools that turn a demo into something you can put a real client in front of.

In [ ]:
!python ep06_indepth.py

batch(2 good)   -> 200  ['setosa', 'virginica']
batch(neg feat) -> 422  (rejected before model)
class/9         -> 404  class index 9 not found
OK: 200 batch, 422 bad input, 404 missing — figure == terminal
wrote sh06.png  (200=['setosa', 'virginica'], 422, 404)


## SH07 · Gradio

A trained model is useless to a human who can't poke at it. FastAPI gave you a machine-readable door; Gradio gives you a human-readable one — a real web UI built straight from your `predict` function, with typed inputs and outputs, no HTML, no JavaScript. The thing that makes this click: Gradio isn't a separate app you wire to your model — it *introspects your function and your component list* to build the page, and you can introspect that same built object back to count exactly what it wired.

In [ ]:
!python ep07_gradio.py

model trained on (150, 4) | classes=3
train accuracy   : 1.000
gradio version   : 6.19.0
inputs wired     : 4
outputs wired    : 1
demo.fn is predict: True
top class for (5.1, 3.5, 1.4, 0.2): setosa (1.000)
to serve: demo.launch()  # shown, not run
OK: figure == terminal (4 inputs -> predict -> 1 output)
wrote sh07.png  (in=4, out=1, top=setosa 1.000)


## SH08 · ONNX

A trained model is trapped inside the library that made it — scikit-learn needs scikit-learn, PyTorch needs PyTorch, every server has to carry the whole training stack just to call `.predict()`. ONNX breaks that chain: you export the model once into a portable graph, and any runtime — C++, Java, a phone, a browser — can run it without ever importing your training code. This episode does the export, runs it through onnxruntime, and proves with `np.allclose` that the portable model agrees with the original down to a millionth.

In [ ]:
!python ep08_onnx.py

trained: LogisticRegression on iris  X=(150, 4)
exported: iris.onnx  (0.7 KB, no Python, no sklearn)
session loaded: input='input'
parity: max|sklearn - onnx| = 1.19e-07
np.allclose(atol=1e-5): True
HEADLINE  size=0.7KB  max_diff=1.19e-07  parity=PASS
wrote sh08.png  (size=0.7KB, max_diff=1.19e-07)


## SH09 · ONNX Optimization

A model that runs is not the same as a model that runs *fast enough*. Dynamic int8 quantization stores your weights as 8-bit integers instead of 32-bit floats, which shrinks the file and speeds up inference — and the price is almost nothing if you measure it honestly. This episode is about proving that claim with your own numbers: size before and after, latency before and after, and an accuracy delta you can read in one decimal.

In [ ]:
!python ep09_quant.py

size   fp32=2.387 MB  int8=0.605 MB  (3.9x smaller)
latency fp32=0.063 ms  int8=0.046 ms  (1.36x faster)
acc    fp32=98.52%  int8=98.52%  (delta +0.00 pp)
figure == terminal :: OK  (saved sh09.png)


## SH10 · vLLM

Serving an LLM is not serving a classifier. A classifier answers in one shot; an LLM writes one token, feeds it back, and writes the next — a loop, with a growing cache of past attention keys and values riding along for every step. The naive way is to run that loop one prompt at a time, and the GPU sits half-idle while it does. vLLM makes the click: PagedAttention stores the KV-cache in fixed-size pages so memory never fragments, and continuous batching keeps swapping finished requests out and new ones in so the GPU is never waiting — and throughput in tokens per second jumps. This episode shows the real vLLM API; the throughput numbers are a representative captured run on a GPU.

In [ ]:
!python ep10_vllm.py

model            : facebook/opt-125m   [representative (no GPU here: ModuleNotFoundError)]
prompts batched  : 8
sample[0] -> 'understanding what a model actually promises'
tokens generated : 1024
wall time (s)    : 2.51
throughput tok/s : 408.0   # representative — varies by GPU
vs naive 1-at-a-time (85 tok/s): 4.8x
OK: 8 prompts -> 8 outputs (batched in a single generate call)
wrote sh10.png  (throughput=408 tok/s, 4.8x, representative)


## SH11 · Embeddings

An embedding turns a sentence into a list of numbers — a point in space — so that meaning becomes geometry. Once text is a vector, "are these two sentences about the same thing?" stops being a hard language problem and becomes a simple distance problem you can compute. Cosine similarity reads the angle between two of those vectors: small angle, close meaning; wide angle, unrelated. This is the quiet machinery under every search box and RAG system you'll ever build.

In [ ]:
!python ep11_embeddings.py

embeddings shape: (6, 384) float64
unit norm check : 1.0

cosine similarity matrix (rounded):
[[ 1.    0.81  0.08  0.08  0.02 -0.  ]
 [ 0.81  1.    0.04  0.05  0.02  0.01]
 [ 0.08  0.04  1.    0.83  0.   -0.01]
 [ 0.08  0.05  0.83  1.    0.03  0.01]
 [ 0.02  0.02  0.    0.03  1.    0.85]
 [-0.    0.01 -0.01  0.01  0.85  1.  ]]

query : 'The cat sat on the mat.'
nearest: 'A feline rested on the rug.'  cosine=0.81
vs cook: 'Whisk the eggs before frying.'  cosine=0.08
vs fin : 'Interest rates rose this quarter.'  cosine=0.02

figure == terminal: heatmap plots this exact matrix.
wrote sh11.png  (nn cosine=0.81, vs cook=0.08, vs fin=0.02)


## SH12 · FAISS

You have a pile of embeddings; the only useful question is *which ones are closest to this one* — and the naive answer, "compare against all of them," is exactly the engine FAISS makes fast and trustworthy. This episode makes one thing click: an `IndexFlatL2` is a brute-force nearest-neighbor search done right — you `add` your matrix once, then `search` a query and get back the top-k indices and their squared distances in milliseconds. We prove it's honest the only way that counts: plant a vector in the index, query a tiny perturbation of it, and watch FAISS return that exact row first, at near-zero distance.

In [ ]:
!python ep12_faiss.py

corpus: xb shape=(10000, 128)  dtype=float32
is_trained: True  (flat index needs no training)
index.ntotal: 10000
search time   : 0.14 ms
top-5 indices: [4242, 7492, 7298, 9764, 3332]
top-5 dists  : [0.012, 13.993, 14.231, 14.859, 14.87]
nearest row    : 4242  (distance 0.012)
HEADLINE  ntotal=10000  top1=4242  dist=0.012  search=0.14ms  PASS
wrote sh12.png  (top1=4242, dist=0.012, search=0.14ms)


## SH13 · FAISS at Scale

Exact nearest-neighbor search is honest but slow — it compares your query against every single vector, every time, and that bill grows with your corpus until a search that felt instant at ten thousand vectors crawls at ten million. Approximate indexes change the deal: by only looking at the most promising slice of the data, they return answers tens of times faster while quietly giving up a sliver of recall. This episode makes that trade visible — you build an exact index for ground truth, build an IVF index beside it, then sweep one knob, `nprobe`, and watch recall@10 climb and query time climb with it. The lesson that clicks: there is no "best" setting, only a curve, and you pick the point on it your latency budget can afford.

In [ ]:
!python ep13_faiss_scale.py

corpus: 100000 vectors, dim=64  |  queries: 1000  |  k=10
flat (exact): 0.746 ms/query  -> ground truth ready
ivf: trained nlist=1024 clusters, added 100000 vectors
 nprobe   recall@10    ms/query   speedup
      1       0.381       0.013       60x
      2       0.639       0.013       57x
      4       0.911       0.019       39x
      8       1.000       0.025       30x
     16       1.000       0.035       21x
     32       1.000       0.058       13x
HEADLINE  nprobe=1 -> recall=0.38@0.01ms  |  nprobe=32 -> recall=1.00@0.06ms
PASS  recall climbs with nprobe; IVF faster than flat
wrote sh13.png  (np1 0.38@0.01ms, np32 1.00@0.06ms, flat 0.75ms)


## SH14 · Weaviate

A raw index gives you nearest neighbors and nothing else. A real vector database wraps that index in the things production actually needs: a typed schema, durable storage, metadata filters, and one query that blends keyword matching with vector similarity. Weaviate is that database, and the move that makes it click is `query.hybrid` — keyword BM25 and vector search fused in a single call, dialed by one number, `alpha`.

In [ ]:
!python ep14_weaviate.py

status: collection 'Doc' created, 5 objects inserted   [representative (no server here: WeaviateConnectionError)]
hybrid hits (category=database, alpha=0.5): 2
  0.81  Vector databases store embeddings
  0.62  Hybrid search blends keyword and vector
filtered out (off-category): 3
HEADLINE  5 inserted -> filter -> 2 ranked -> top 0.81  (hybrid alpha=0.5)
wrote sh14.png  (top 0.81, 2nd 0.62, representative)


## SH15 · RAG

A language model only knows what it absorbed in training, so when you ask about your own documents it makes things up with total confidence. Retrieval-Augmented Generation fixes that by doing a search first: chunk your docs, embed them, find the top-k chunks that actually match the question, and paste those into the prompt before the model answers. The retrieval is real engineering you fully control — FAISS finds the right chunk every time — and the model's job shrinks from "know everything" to "read these three paragraphs and answer." When the answer can name the chunk it came from, hallucination stops being a guess and starts being a citation.

In [ ]:
!python ep15_rag.py

question      : How do I roll back a deployment?
chunks indexed: 8
  retrieved   : rollback-doc#0 cos=0.562
  retrieved   : deploy-doc#0   cos=0.383
  retrieved   : logging-doc#1  cos=0.162
top chunk     : rollback-doc#0
answer        : "To roll back a deployment, run kubectl rollout undo."
cited source  : rollback-doc
OK: cited source == true source
wrote sh15.png  (top rollback-doc#0 cos=0.562, cited rollback-doc)


## SH16 · From RAG to Graph RAG

Flat retrieval finds chunks that *look* like your question, but it has no idea how those chunks connect. A knowledge graph stores the connections directly — entities as nodes, relationships as edges — so the system can know that Alice built Service A, Service A depends on the Postgres cluster, and the Postgres cluster lives in the eu-west region, all as one traversable chain. This episode builds that graph from a tiny set of seeded triples with networkx and reports its shape, then walks a three-hop path that no single chunk in a vector index could ever contain.

In [ ]:
!python ep16_graph.py

nodes: 8
edges: 8
busiest hub: postgres-cluster (degree 3)
path Alice -> eu-west: Alice -> Service-A -> postgres-cluster -> eu-west
hops: 3
OK figure == terminal  (wrote sh16.png)


## SH17 · Graph RAG Retrieval

Not every question is the same shape, so not every retrieval should be either. A *local* question — "tell me about this one thing" — wants the neighborhood around an entity, which you get by walking the graph a few hops out. A *global* question — "what are the big themes here?" — wants a summary of the clusters, which you get by detecting communities and describing each one. Graph RAG is just knowing which of those two walks to take.

In [ ]:
!python ep17_graphrag.py

graph: 12 nodes 17 edges

LOCAL  'Transformer' (2-hop neighborhood): 6 nodes
  neighborhood: ['Attention', 'BERT', 'FastAPI', 'GPT', 'RAG', 'Transformer']
  multi-hop 'Transformer' -> 'RAG': hops=2
  path: Transformer -> GPT -> RAG

GLOBAL detected 3 communities
  [Model architecture] The Model architecture community covers 4 entities: ['Attention', 'BERT', 'GPT', 'Transformer'].
  [Serving & infra] The Serving & infra community covers 4 entities: ['Docker', 'FastAPI', 'Kubernetes', 'Uvicorn'].
  [Retrieval] The Retrieval community covers 4 entities: ['Embeddings', 'FAISS', 'RAG', 'Weaviate'].

figure == terminal: 3 communities of 4, path hops = 2
wrote sh17.png  (12 nodes, 3 communities of 4, path hops 2)


## SH18 · Capstone — The Full Stack

Every stage of this course was a single promise kept in isolation; the capstone is the moment they have to keep their promises *together*, in one program, in one run. We build a `build_pipeline()` that walks the whole arc — an MLflow-tracked artifact, served behind a FastAPI endpoint, optimized through ONNX with a parity check, answering a question through a FAISS-plus-graph retrieval layer — and we assert each stage's headline number so the proof is the green run, not a slide. The thing that clicks: the course *composes*. Five tools you learned one at a time turn out to be five links of a chain that carries a real request from notebook to answer.

In [ ]:
!python ep18_capstone.py

PACKAGE   {'run_id': 'da5b3e0a', 'acc': 1.0}
SERVE     {'code': 200, 'pred': 0}
OPTIMIZE  {'parity': True, 'maxdiff': 6.816578473900492e-08}
RETRIEVE  {'hit': 'c3', 'hops': 3}
SHIP      {'answer': 'Alice built Service-A; chain: Alice -> Service-A -> postgres-cluster -> eu-west'}
5/5 stages green   (ONNX max|diff| = 6.8e-08)
wrote sh18.png  (5/5 green, hit c3, 3 hops)
